# Suite FRM — The framework

Every dialect implements the same protocols (`@mbse/expressions/Framework`): its terms are values of the framework's
`Term`, the union of the dialects' `Term`s, stored through its install, structurally traversable through forms, validatable, evaluatable and
translatable. `Framework.Terms` reads any dialect's terms by its kinds (`Terms.Kind`, `Terms.Dialect`); this suite
checks that machinery on every dialect at once, and on an extension of Patterns declared here. The dialects' own rules are
suite DIA, and translation suite TRN.

In [ ]:
import { Framework } from "@mbse/expressions";
import { Simplifiers as V, Expressions as E } from "@mbse/expressions/Patterns/Expressions";
import * as IEEE1164 from "@mbse/expressions/IEEE1164/Expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as Unicode from "@mbse/expressions/Unicode/Expressions";
import * as Patterns from "@mbse/expressions/Patterns/Expressions";
import * as CDialect from "@mbse/expressions/C/Expressions";
import * as CppDialect from "@mbse/expressions/Cpp/Expressions";
import * as Excel from "@mbse/expressions/Excel/Expressions";
import * as Latex from "@mbse/expressions/LaTeX/Expressions";
import * as Matlab from "@mbse/expressions/MATLAB/Expressions";
import * as Python3 from "@mbse/expressions/Python/Python3";
import * as AAPCS64 from "@mbse/expressions/Arm/AAPCS64";
import * as AMD64 from "@mbse/expressions/SystemV/AMD64";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as SVAssertions from "@mbse/expressions/SystemVerilog/Assertions";
import * as SVConstraints from "@mbse/expressions/SystemVerilog/Constraints";
import * as SVPatterns from "@mbse/expressions/SystemVerilog/Patterns";
import * as SVPreprocessor from "@mbse/expressions/SystemVerilog/Preprocessor";
import * as SVTypes from "@mbse/expressions/SystemVerilog/Types";
import * as TypeScript5 from "@mbse/expressions/TypeScript/TypeScript5";
import * as CPreprocessor from "@mbse/expressions/C/Preprocessor"; // the retrofit's new dialects
import * as CTypes from "@mbse/expressions/C/Types";
import * as CppRegExp from "@mbse/expressions/Cpp/RegExp"; // and the regular expressions std::regex reads
import * as CppTypes from "@mbse/expressions/Cpp/Types";
import * as PosixBRE from "@mbse/expressions/POSIX/BRE"; // and POSIX's, which a C++ regex may hold
import * as PosixERE from "@mbse/expressions/POSIX/ERE";
import * as PythonPatterns from "@mbse/expressions/Python/Patterns"; // the retrofit's, built
import * as PythonRegExp from "@mbse/expressions/Python/RegExp";
import * as PythonTypes from "@mbse/expressions/Python/Types";
import * as TypeScriptPatterns from "@mbse/expressions/TypeScript/Patterns";
import * as TypeScriptRegExp from "@mbse/expressions/TypeScript/RegExp";
import * as TypeScriptTypes from "@mbse/expressions/TypeScript/Types";
import { Domains as D, Errors as FE, Simplifiers as FV, Standards as FS, Symbolics as SY, Terms as F, Translators as FT } from "@mbse/expressions/Framework";
import * as Translators from "@mbse/expressions/Translators";
import { Builders, Errors, JSON, Repr, Validators, YAML } from "@mbse/schemas/Framework";
import { assert, conformance_problems, equal, form_equal, map, raises, same } from "./support.js";
import { Simplifiers as Simplification } from "@mbse/expressions/Framework";
const { NotEvaluated } = Simplification;

const BD = Patterns.Domains;
const [P, M, X, L, C, CPP, SV] = [Python3.Expressions, Matlab.Expressions, Excel.Expressions, Latex.Expressions,
  CDialect.Expressions, CppDialect.Expressions, SystemVerilog.Expressions];
const TS = TypeScript5.Expressions;
const [PYT, PYP, PYR] = [PythonTypes.Expressions, PythonPatterns.Expressions, PythonRegExp.Expressions];
const [TST, TSP, TSR] = [TypeScriptTypes.Expressions, TypeScriptPatterns.Expressions, TypeScriptRegExp.Expressions];
const [SVT, SVP, SVA, SVC, SVPP] = [SVTypes.Expressions, SVPatterns.Expressions, SVAssertions.Expressions,
  SVConstraints.Expressions, SVPreprocessor.Expressions];
const { KeyError, NotImplementedError, ValueError } = Errors;

const RETROFIT: any[] = [PythonTypes, PythonPatterns, TypeScriptTypes, TypeScriptPatterns, CTypes, CppTypes, CPreprocessor, // installed so
  PythonRegExp, TypeScriptRegExp, CppRegExp, PosixBRE, PosixERE]; // the regular expression dialects last, POSIX's after
// C++'s
const store: any = new Builders.Store();
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, IEEE1164.install, Unicode.install, // the standard
  Patterns.install, Python3.install, Matlab.install, Excel.install, Latex.install, // dialects first
  CDialect.install, CppDialect.install, AMD64.install, AAPCS64.install, SystemVerilog.install, SVTypes.install,
  SVPatterns.install, SVAssertions.install, SVConstraints.install, SVPreprocessor.install, TypeScript5.install,
  ...RETROFIT.map((pkg) => pkg.install)]) { // each after its family's dialect
  store.transaction(install);
}
store.transaction(Patterns.define); // Patterns' operations, which its validation reads
for (const define of [CDialect.define, CppDialect.define, AMD64.define, AAPCS64.define]) { // C's, C++'s and the ABIs' types,
  store.transaction(define); // which their evaluation, inference and validation read through a term's holder
}
store.transaction(SystemVerilog.define); // SystemVerilog's types, which its evaluation, inference and validation read
for (const define of [SVTypes.define, SVPatterns.define, SVAssertions.define, SVConstraints.define, SVPreprocessor.define]) {
  store.transaction(define); // the editions SystemVerilog's sublanguages cover
}
for (const pkg of RETROFIT) store.transaction(pkg.define); // the retrofit's new dialects' data: the editions they cover, or none yet
const Term: any = store.registered("Expressions.Framework.Term");
const AMD64_ = store.Instances.Expressions.SystemV.AMD64; // the ABI a C or C++ term's holder names

/** The term value `spec` writes, the root of a holder written in a transaction; a C or C++ term's holder names the ABI
 * its terms are read under, x86-64's System V ABI. */
function term(spec: unknown, dialect: F.Dialect = E.DIALECT): any {
  return F.term(store.transaction(() => store.Expressions.Framework.Expression((e: any) =>
    (["c", "cpp"].includes(dialect.tag) ? e.root(dialect.spec(spec)).abi([AMD64_]) : e.root(dialect.spec(spec))))));
}

// TypeScript5 has no writers: its terms are written through the builders
const tsPerson = (u: any) => u.identifier((r: any) => r.name("person")); // TypeScript5's `tsPerson`, in any union that has identifiers
const typescript5 = (u: any) => u.sequenceexpression((s: any) => s.expressions([ // every kind, written through the builders
    (e: any) => e.parenthesizedexpression((r: any) => r.expression((x: any) => x.assignmentexpression((a: any) => a  // ({ a: [tsPerson, , ...tsPerson], d: { tsPerson }, tsPerson = 1 } = tsPerson),
        // a destructuring assignment's targets: references to what it assigns
        .left((t: any) => t.objectpattern((o: any) => o.properties([
            (p: any) => p.property((q: any) => q.propertyKind("init").key((k: any) => k.identifier((i: any) => i.name("a"))).value(
                (v: any) => v.arraypattern((ap: any) => ap.elements([tsPerson, (y: any) => y.elision((h: any) => h),
                                                                  (y: any) => y.restelement((z: any) => z.argument(tsPerson))])))),
            (p: any) => p.property((q: any) => q.propertyKind("init").key((k: any) => k.identifier((i: any) => i.name("d"))).value(
                (v: any) => v.objectpattern((op: any) => op.properties([(y: any) => y.property((z: any) => z.propertyKind("init").key(tsPerson).shorthand(true).value(tsPerson))])))),
            (p: any) => p.property((q: any) => q.propertyKind("init").key(tsPerson).shorthand(true).value(
                (v: any) => v.assignmentpattern((ap: any) => ap.left(tsPerson).right(
                    (g: any) => g.literal((li: any) => li.value((n: any) => n.as_scalar(1.0)))))))])))
        .operator("=").right(tsPerson)))),
    (e: any) => e.functionexpression((r: any) => r.isAsync(true).generator(true).id((i: any) => i.name("g")).typeParameters(  // async function* g<T>(x: T = this.y) {}
        (d: any) => d.params([(p: any) => p.name((i: any) => i.name("T"))])).params([(p: any) => p.assignmentpattern((ap: any) => ap
            .left((g: any) => g.identifier((i: any) => i.name("x").typeAnnotation((ta: any) => ta.typeAnnotation(
                (t: any) => t.tstypereference((tr: any) => tr.typeName((n: any) => n.identifier((j: any) => j.name("T"))))))))
            .right((g: any) => g.memberexpression((m: any) => m.object((o: any) => o.thisexpression((h: any) => h)).property((y: any) => y.identifier((i: any) => i.name("y"))))))])),
    (e: any) => e.arrowfunctionexpression((r: any) => r.isAsync(true).typeParameters((d: any) => d.params([(p: any) => p.name((i: any) => i.name("U"))]))  // async <U>(p: U, ...r: U[]) => await import.meta.resolve(p)
        .params([(p: any) => p.identifier((i: any) => i.name("p").typeAnnotation((ta: any) => ta.typeAnnotation(
                     (t: any) => t.tstypereference((tr: any) => tr.typeName((n: any) => n.identifier((j: any) => j.name("U"))))))),
                 (p: any) => p.restelement((z: any) => z.argument((g: any) => g.identifier((i: any) => i.name("r"))).typeAnnotation((ta: any) => ta.typeAnnotation(
                     (t: any) => t.tsarraytype((at: any) => at.elementType((y: any) => y.tstypereference((tr: any) => tr.typeName((n: any) => n.identifier((j: any) => j.name("U")))))))))])
        .body((b: any) => b.awaitexpression((w: any) => w.argument((c: any) => c.callexpression((ce: any) => ce.callee((f: any) => f.memberexpression((m: any) => m
            .object((o: any) => o.metaproperty((mp: any) => mp.meta((i: any) => i.name("import")).property((i: any) => i.name("meta"))))
            .property((y: any) => y.identifier((i: any) => i.name("resolve"))))).arguments([(y: any) => y.identifier((i: any) => i.name("p"))])))))),
    (e: any) => e.classexpression((r: any) => r.decorators([(d: any) => d.expression(tsPerson)]).id((i: any) => i.name("K"))  // @tsPerson class K<T> extends tsPerson.Base<T> implements tsPerson {}
        .typeParameters((d: any) => d.params([(p: any) => p.name((i: any) => i.name("T"))]))
        .superClass((c: any) => c.memberexpression((m: any) => m.object(tsPerson).property((y: any) => y.identifier((i: any) => i.name("Base")))))
        .superTypeArguments((a: any) => a.params([(t: any) => t.tstypereference((tr: any) => tr.typeName((n: any) => n.identifier((j: any) => j.name("T"))))]))
        .implements([(c: any) => c.expression(tsPerson)])),
    (e: any) => e.newexpression((r: any) => r.callee((c: any) => c.memberexpression((m: any) => m.object(tsPerson).property(  // new tsPerson.C<number>(...tsPerson)
        (y: any) => y.identifier((i: any) => i.name("C"))))).typeArguments((a: any) => a.params([(t: any) => t.tsnumberkeyword((k: any) => k)]))
        .arguments([(y: any) => y.spreadelement((z: any) => z.argument(tsPerson))])),
    (e: any) => e.taggedtemplateexpression((r: any) => r.tag(tsPerson).quasi((q: any) => q.quasis([  // tsPerson`a${tsPerson}b`
        (y: any) => y.raw("a"), (y: any) => y.raw("b").tail(true)]).expressions([tsPerson]))),
    (e: any) => e.templateliteral((r: any) => r.quasis([(y: any) => y.raw("x"), (y: any) => y.raw("").tail(true)]).expressions([tsPerson])),  // `x${tsPerson}`
    (e: any) => e.updateexpression((r: any) => r.operator("++").argument(tsPerson)),  // tsPerson++
    (e: any) => e.unaryexpression((r: any) => r.operator("typeof").argument(tsPerson)),  // typeof tsPerson
    (e: any) => e.binaryexpression((r: any) => r.left((y: any) => y.privateidentifier((i: any) => i.name("p"))).operator("in").right(tsPerson)),  // #p in tsPerson
    (e: any) => e.logicalexpression((r: any) => r.left(tsPerson).operator("??").right((y: any) => y.constant((c: any) => c.singleton("null")))),  // tsPerson ?? null
    (e: any) => e.conditionalexpression((r: any) => r.test(tsPerson).consequent(  // tsPerson ? /x/g : 1n
        (y: any) => y.constant((c: any) => c.pattern("x").flags("g"))).alternate((y: any) => y.literal((li: any) => li.value((n: any) => n.as_scalar(1n))))),
    (e: any) => e.yieldexpression((r: any) => r.delegate(true).argument(tsPerson)),  // yield* tsPerson
    (e: any) => e.tsasexpression((r: any) => r.expression(tsPerson).typeAnnotation((t: any) => t.tsstringkeyword((k: any) => k))),  // tsPerson as string
    (e: any) => e.tssatisfiesexpression((r: any) => r.expression(tsPerson).typeAnnotation((t: any) => t.tsobjectkeyword((k: any) => k))),  // tsPerson satisfies object
    (e: any) => e.tstypeassertion((r: any) => r.typeAnnotation((t: any) => t.tsanykeyword((k: any) => k)).expression(tsPerson)),  // <any>tsPerson
    (e: any) => e.tsnonnullexpression((r: any) => r.expression(tsPerson)),  // tsPerson!
    (e: any) => e.tsinstantiationexpression((r: any) => r.expression(tsPerson).typeArguments((a: any) => a.params([(t: any) => t.tsstringkeyword((k: any) => k)]))),  // tsPerson<string>
    (e: any) => e.arrayexpression((r: any) => r.elements([  // [1, , ...tsPerson]: the framework's list, its elements any dialect's terms
        (y: any) => y.typescript5((t: any) => t.literal((li: any) => li.value((n: any) => n.as_scalar(1.0)))), (y: any) => y.typescript5((t: any) => t.elision((h: any) => h)),
        (y: any) => y.typescript5((t: any) => t.spreadelement((z: any) => z.argument(tsPerson)))])),
    (e: any) => e.objectexpression((r: any) => r.properties([  // { m(): void, ...tsPerson, [tsPerson]: "s" }
        (p: any) => p.property((q: any) => q.propertyKind("init").key((k: any) => k.identifier((i: any) => i.name("m"))).method(true).value(
            (v: any) => v.tsemptybodyfunctionexpression((f: any) => f.returnType((a: any) => a.typeAnnotation((t: any) => t.tsvoidkeyword((k: any) => k)))))),
        (p: any) => p.spreadelement((z: any) => z.argument(tsPerson)),
        (p: any) => p.property((q: any) => q.propertyKind("init").computed(true).key(tsPerson).value((v: any) => v.literal((li: any) => li.value((n: any) => n.as_scalar("s")))))])),
    (e: any) => e.jsxelement((r: any) => r  // <div id="x" {...tsPerson}>{}{tsPerson}<>text</></div>
        .openingElement((o: any) => o.name((n: any) => n.jsxidentifier((i: any) => i.name("div"))).attributes([
            (a: any) => a.jsxattribute((t: any) => t.name((n: any) => n.jsxidentifier((i: any) => i.name("id"))).value(
                (v: any) => v.literal((li: any) => li.value((n: any) => n.as_scalar("x")).text('"x"')))),
            (a: any) => a.jsxspreadattribute((z: any) => z.argument(tsPerson))]))
        .children([(c: any) => c.jsxexpressioncontainer((x: any) => x.expression((y: any) => y.jsxemptyexpression((h: any) => h))),
                   (c: any) => c.jsxexpressioncontainer((x: any) => x.expression(tsPerson)),
                   (c: any) => c.jsxfragment((f: any) => f.openingFragment((h: any) => h).children([(t: any) => t.jsxtext((j: any) => j.raw("text"))]).closingFragment((h: any) => h))])
        .closingElement((c: any) => c.name((n: any) => n.jsxidentifier((i: any) => i.name("div"))))),
    (e: any) => e.jsxfragment((f: any) => f.openingFragment((h: any) => h).children([(c: any) => c.jsxelement((x: any) => x  // <><br /></>
        .openingElement((o: any) => o.name((n: any) => n.jsxidentifier((i: any) => i.name("br"))).selfClosing(true)))]).closingFragment((h: any) => h)),
    (e: any) => e.chainexpression((r: any) => r.expression((y: any) => y.memberexpression((m: any) => m.object(tsPerson).optional(true).property(  // tsPerson?.a
        (z: any) => z.identifier((i: any) => i.name("a")))))),
    (e: any) => e.memberexpression((m: any) => m.object((o: any) => o.super((h: any) => h)).property((y: any) => y.identifier((i: any) => i.name("x")))),  // super.x
    (e: any) => e.importexpression((r: any) => r.source(tsPerson)),  // import(tsPerson)
]));
const person = E.variable("person");
const [cPerson, cppPerson] = [C.identifier("person"), CPP.identifier("person")];
const cInt = (t: any) => t.typeid((i: any) => i.specifiers([(s: any) => s.primitivetypespecifier((p: any) => p.keyword("int"))])); // `int`, in C's or C++'s
const uint8 = (t: any) => t.typeid((i: any) => i.specifiers([(s: any) => s.namedtypespecifier((n: any) => n.name( // `uint8_t`, in C's or C++'s
  (m: any) => m.identifier((d: any) => d.spelling("uint8_t"))))]));
const cEvery = (u: any) => u.initializerlist((r: any) => r.items([ // every kind of C's, written through the builders and its writers
  C.conditional(C.binary("&&", C.unary("!", C.member(cPerson, "a")), C.binary(">", C.subscript(C.member(cPerson, "b", "->"), 0n), C.literal(1.5, "f"))),
                C.cast(uint8, C.call("person", 1n)), C.literal(5n, "u")),  // the kinds C evaluates
  (i: any) => i.constant((c: any) => c.singleton("nullptr")),  // nullptr
  (i: any) => i.concatenatedstring((c: any) => c.parts([C.literal("a"), C.literal("b")])),  // "a" "b"
  (i: any) => i.parenthesizedexpression((p: any) => p.expression(cPerson)),  // (person)
  (i: any) => i.genericselection((g: any) => g.controlling(cPerson).associations([  // _Generic(person, int: 1, default: 0)
      (a: any) => a.type(cInt).value(C.literal(1n)), (a: any) => a.value(C.literal(0n))])),
  (i: any) => i.postfixexpression((p: any) => p.operand(cPerson).operator("++")),  // person++
  (i: any) => i.sizeofexpression((s: any) => s.operand(cInt)),  // sizeof(int): a type name where an expression may stand
  (i: any) => i.alignofexpression((s: any) => s.keyword("_Alignof").operand(cPerson)),  // _Alignof(person)
  (i: any) => i.assignmentexpression((a: any) => a.left(cPerson).operator("+=").right(C.literal(1n))),  // person += 1
  (i: any) => i.designatedinitializer((d: any) => d.designators([  // .x[0] = person
      (x: any) => x.fielddesignator((f: any) => f.name((j: any) => j.spelling("x"))), (x: any) => x.indexdesignator((f: any) => f.index(C.literal(0n)))]).initializer(
      (x: any) => x.equalinitializer((e: any) => e.value(cPerson)))),
  (i: any) => i.compoundliteralexpression((c: any) => c.type((t: any) => t.typeid((y: any) => y.specifiers([  // (int[]){1}
      (s: any) => s.primitivetypespecifier((p: any) => p.keyword("int"))]).declarator((d: any) => d.arraydeclarator((a: any) => a)))).initializer(
      (li: any) => li.items([C.literal(1n)]))),
  (i: any) => i.statementexpression((s: any) => s),  // ({}): GNU's, its body of statements not part of the term
  (i: any) => i.extensionexpression((e: any) => e.operand(cPerson)),  // __extension__ person
]));
const cppName = (spelling: string) => (n: any) => n.identifier((i: any) => i.spelling(spelling)); // a C++ name that is an identifier
const cppEvery = (u: any) => u.initializerlist((r: any) => r.items([ // every kind of C++'s, written through the builders and its writers
  CPP.conditional(CPP.binary("&&", CPP.unary("!", CPP.member(cppPerson, "a")), CPP.binary(">", CPP.subscript(CPP.member(cppPerson, "b", "->"), 0n), CPP.literal(1.5, "f"))),
                  CPP.cast(uint8, CPP.call("person", 1n)), CPP.literal(5n, "u")),  // the kinds C++ evaluates
  (i: any) => i.constant((c: any) => c.singleton("nullptr")),  // nullptr
  (i: any) => i.userdefinedliteral((d: any) => d.literal((li: any) => li.value((v: any) => v.as_scalar(10n))).suffix("_km")),  // 10_km
  (i: any) => i.concatenatedstring((c: any) => c.parts([CPP.literal("a"), (x: any) => x.literal(  // "a" R"(b)"
      (li: any) => li.value((v: any) => v.as_scalar("b")).raw(true))])),
  (i: any) => i.idexpression((d: any) => d.name((n: any) => n.qualifiedname((q: any) => q.qualifiers([cppName("std")]).name(cppName("cout"))))),  // std::cout
  (i: any) => i.thisexpression((t: any) => t),  // this
  (i: any) => i.parenthesizedexpression((x: any) => x.expression(cppPerson)),  // (person)
  (i: any) => i.lambdaexpression((la: any) => la.captures([  // [=, &person](auto y) {}: its body of statements not part of the term
      (c: any) => c.defaultcapture((d: any) => d.mode("=")), (c: any) => c.simplecapture((s: any) => s.by_reference(true).name((j: any) => j.spelling("person")))]).declarator(
      (d: any) => d.parameters([(q: any) => q.parameterdeclaration((pd: any) => pd.specifiers([(s: any) => s.placeholdertypespecifier((h: any) => h)]).declarator(
          (dd: any) => dd.iddeclarator((i2: any) => i2.name(cppName("y")))))]))),
  (i: any) => i.foldexpression((f: any) => f.left(cppPerson).operator("+")),  // (person + ...)
  (i: any) => i.requiresexpression((q: any) => q.parameters([(a: any) => a.parameterdeclaration(  // requires (int z) { z + 1; }
      (pd: any) => pd.specifiers([(s: any) => s.primitivetypespecifier((k: any) => k.keyword("int"))]).declarator(
          (dd: any) => dd.iddeclarator((i2: any) => i2.name(cppName("z")))))]).requirements([
      (r2: any) => r2.simplerequirement((s: any) => s.expression(CPP.binary("+", CPP.identifier("z"), 1n)))])),
  (i: any) => i.functionalcastexpression((f: any) => f.type((s: any) => s.primitivetypespecifier((q: any) => q.keyword("int"))).initializer(  // int(person)
      (x: any) => x.parenthesizedinitializer((pi: any) => pi.arguments([cppPerson])))),
  (i: any) => i.postfixexpression((x: any) => x.operand(cppPerson).operator("--")),  // person--
  (i: any) => i.namedcastexpression((n: any) => n.operator("static_cast").type(cInt).operand(cppPerson)),  // static_cast<int>(person)
  (i: any) => i.typeidexpression((t: any) => t.operand(cppPerson)),  // typeid(person)
  (i: any) => i.awaitexpression((a: any) => a.operand(cppPerson)),  // co_await person
  (i: any) => i.sizeofexpression((s: any) => s.operand(cInt)),  // sizeof(int)
  (i: any) => i.sizeofpackexpression((s: any) => s.pack((j: any) => j.spelling("Ts"))),  // sizeof...(Ts)
  (i: any) => i.alignofexpression((s: any) => s.keyword("alignof").operand(cInt)),  // alignof(int)
  (i: any) => i.noexceptexpression((s: any) => s.operand(cppPerson)),  // noexcept(person)
  (i: any) => i.newexpression((n: any) => n.type(cInt).initializer((x: any) => x.initializerlist((il: any) => il.items([CPP.literal(1n)])))),  // new int{1}
  (i: any) => i.deleteexpression((d: any) => d.array(true).operand(cppPerson)),  // delete[] person
  (i: any) => i.assignmentexpression((a: any) => a.left(cppPerson).operator("=").right(CPP.literal(0n))),  // person = 0
  (i: any) => i.throwexpression((t: any) => t),  // throw
  (i: any) => i.yieldexpression((y: any) => y.operand(cppPerson)),  // co_yield person
  (i: any) => i.packexpansion((x: any) => x.pattern(cppPerson)),  // person...
  (i: any) => i.designatedinitializer((d: any) => d.designators([(x: any) => x.fielddesignator((f: any) => f.name((j: any) => j.spelling("x")))]).initializer(  // .x = person
      (x: any) => x.equalinitializer((e: any) => e.value(cppPerson)))),
  (i: any) => i.statementexpression((s: any) => s),  // ({})
  (i: any) => i.extensionexpression((e: any) => e.operand(cppPerson)),  // __extension__ person
]));
const [CT, XT, CPRE] = [CTypes.Expressions.DIALECT, CppTypes.Expressions.DIALECT, CPreprocessor.Expressions.DIALECT];
const XR = CppRegExp.Expressions.DIALECT;
const C_FAMILY = [CT, XT, CPRE, XR]; // the C family's new dialects, built: C's and C++'s types, the preprocessor both read,
// and C++'s regular expressions
const POSIX = [PosixBRE.Expressions.DIALECT, PosixERE.Expressions.DIALECT]; // POSIX's regular expressions, which a C++ regex
// naming a POSIX grammar holds
const cSpelled = (spelling: string) => (i: any) => i.spelling(spelling); // an identifier, C's or C++'s

/** Every kind of a Types dialect: an enumeration's type name, its body's enumerators, and a directive and a comment among
 * them, C Preprocessor's: `enum E { A = 1, #ifdef B_ B, #endif //last }`. */
function enumeration(types: F.Dialect, family: any): (t: any) => any {
  return (t: any) => t.typeid((i: any) => i.specifiers([(s: any) => s.enumspecifier((e: any) => e.key("enum").name(
    (n: any) => n.identifier(cSpelled("E"))).body(true).enumerators([
      (n: any) => n.enumerator((r: any) => r.name(cSpelled("A")).value(family.literal(1n))),
      (n: any) => n.ifdefdirective((d: any) => d.name(cSpelled("B_")).items([types.spec((m: any) => m.enumerator(
        (r: any) => r.name(cSpelled("B"))))])),
      (n: any) => n.comment((c: any) => c.text("last").trailing(true))]).trailing_comma(true))]));
}

const cPreprocessorEvery = (u: any) => u.ifdirective((d: any) => d.condition( // every kind of C Preprocessor's
  C.binary("&&", C.DIALECT.spec((v: any) => v.definedexpression((x: any) => x.name(cSpelled("X")))), C.binary( // a C condition
    ">", C.identifier("X"), 1n))).items([CT.spec((m: any) => m.enumerator((r: any) => r.name(cSpelled("A")))), // its items
  CPRE.spec((m: any) => m.includedirective((i: any) => i.path("a.h").system(true))),
  CPRE.spec((m: any) => m.comment((c: any) => c.block(true).text(" one ")))]).alternative(
  (a: any) => a.elifdirective((e: any) => e.condition(CPP.binary("==", CPP.identifier("Y"), 2n)).alternative( // a C++ one
    (b: any) => b.elifdefdirective((f: any) => f.negated(true).name(cSpelled("Z")).alternative(
      (c: any) => c.elsedirective((g: any) => g.items([
        CPRE.spec((m: any) => m.definedirective((h: any) => h.name(cSpelled("F")).function_like(true).parameters([
          cSpelled("a")]).variadic(true).replacement("a + 1"))),
        CPRE.spec((m: any) => m.otherdirective((o: any) => o.directive("pragma").text("once"))),
        CPRE.spec((m: any) => m.ifdefdirective((o: any) => o.name(cSpelled("W"))))]))))))));
const basic = E.let_("a", person.age, E.variable("a").ge(18n).and_(person.has("email")));
// SystemVerilog's: every kind of its Expressions and its helpers, with its Types and the Preprocessor's macro usage
// written through the builders where a writer has none
const svPerson = SV.identifier("person");
const svIdentifier = (spelling: string) => (i: any) => i.spelling(spelling); // an identifier, of its spelling
const svMember = (u: any) => u.memberexpression((m: any) => m.value(svPerson).member(svIdentifier("a"))); // person.a, in any union
const svAtom = (keyword: string) => (t: any) => t.integeratomtype((a: any) => a.keyword(keyword)); // `int`, a Types term
const svEvery = SV.binary("->", SV.inside(SV.member(svPerson, "a"), 1n, SV.span(2n, 5n)), SV.concatenation(
    (u: any) => u.conditionalexpression((c: any) => c.condition((m: any) => m.matchesexpression((x: any) => x.value(svPerson).pattern(SV.literal(3n))))
        .attributes([(a: any) => a.specs([(s: any) => s.name(svIdentifier("keep")).value(SV.literal(1n))])])
        .consequence((q: any) => q.literal((r: any) => r.value((v: any) => v.as_scalar(31n)).base("h"))).alternative(SV.vector("10x1", true))),
    SV.select(svPerson, 0n), SV.range_(svPerson, 7n, 4n), SV.unary("-", svPerson),
    (u: any) => u.incrementexpression((i: any) => i.operator("++").postfix(true).operand(svPerson)),
    (u: any) => u.parenthesizedexpression((p: any) => p.expression((e: any) => e.assignmentexpression((a: any) => a.target(svPerson).operator("+=").value(SV.literal(1n))))),
    SV.replication(2n, svPerson),
    (u: any) => u.assignmentpattern((a: any) => a.type(svAtom("int")).items([(i: any) => i.patternitem((p: any) => p.key(svPerson).value(SV.literal(1n))),
                                                                          (i: any) => i.patternitem((p: any) => p.value(SV.literal(0n)))])),
    (u: any) => u.assignmentpattern((a: any) => a.count(SV.literal(2n)).items([svPerson])),
    (u: any) => u.thisexpression((t: any) => t), (u: any) => u.superexpression((t: any) => t),
    (u: any) => u.newexpression((n: any) => n.scope((s: any) => s.scopedname((c: any) => c.scope((x: any) => x.identifier(svIdentifier("pkg"))).name(
        (x: any) => x.parameterizedname((q: any) => q.name(svIdentifier("c")).parameters([SV.literal(8n)]))))).arguments([
        SV.literal(1n), (a: any) => a.namedconnection((c: any) => c.name(svIdentifier("n")).value(svPerson)), (a: any) => a.emptyargument((e: any) => e)])),
    (u: any) => u.newexpression((n: any) => n.scope((s: any) => s.superexpression((x: any) => x)).defaulted(true)),
    (u: any) => u.newcopyexpression((n: any) => n.value(svPerson)), (u: any) => u.newarrayexpression((n: any) => n.size(SV.literal(4n)).value(svPerson)),
    (u: any) => u.randomizewithexpression((r: any) => r.call(SV.method(svPerson, "randomize")).restricted(true).variables([svIdentifier("a")]).items(
        [SV.binary("<", svPerson, 5n)])),
    SV.iterate(svPerson, "sum", "q", SV.binary(">", SV.identifier("q"), 0n)),
    (u: any) => u.callexpression((c: any) => c.callee(svPerson).attributes([(a: any) => a.specs([(s: any) => s.name(svIdentifier("pure"))])]).arguments([
        svPerson, (a: any) => a.namedconnection((n: any) => n.name(svIdentifier("n")).implicit(true)), (a: any) => a.emptyargument((e: any) => e)])),
    SV.call("$bits", svAtom("int")), SV.call("$clog2", svPerson), SV.cast(svAtom("byte"), svPerson), SV.cast(8n, svPerson),
    (u: any) => u.dollarexpression((d: any) => d), (u: any) => u.rootexpression((r: any) => r), (u: any) => u.emptyqueue((e: any) => e),
    (u: any) => u.mintypmaxexpression((m: any) => m.min(SV.literal(1n)).typ(SV.literal(2n)).max(SV.literal(3n))),
    (u: any) => u.taggedexpression((t: any) => t.member(svIdentifier("Valid")).value(svPerson)),
    (u: any) => u.predicateexpression((p: any) => p.conditions([(c: any) => c.matchesexpression((m: any) => m.value(svPerson).pattern(SV.literal(1n))), svPerson])),
    (u: any) => u.streamingconcatenation((s: any) => s.operator("<<").slice(svAtom("byte")).items([
        (i: any) => i.expression(svPerson).left(SV.literal(0n)).operator(":").right(SV.literal(1n))])),
    (u: any) => u.nameexpression((n: any) => n.name((x: any) => x.localname((l: any) => l.name(svIdentifier("y"))))),
    (u: any) => u.nameexpression((n: any) => n.name((x: any) => x.scopedname((c: any) => c.scope((y: any) => y.unitname((z: any) => z)).name((y: any) => y.identifier(svIdentifier("z")))))),
    (u: any) => u.nameexpression((n: any) => n.name((x: any) => x.interfacetypename((i: any) => i.interface(svIdentifier("bus")).name(svIdentifier("t"))))),
    (u: any) => u.nameexpression((n: any) => n.name((x: any) => x.interfacemethodname((i: any) => i.port(svIdentifier("port")).name(svIdentifier("m"))))),
    (u: any) => u.macrousage((m: any) => m.name(svIdentifier("WIDTH"))), (u: any) => u.constant((c: any) => c.fill("1")),
    (u: any) => u.constant((c: any) => c.singleton("null")), SV.literal(1.5)));
const typesEvery = (u: any) => u.structtype((s: any) => s.keyword("union").qualifier("tagged").packed(true).signing("signed").members([
    (m: any) => m.random("rand").type((t: any) => t.integervectortype((v: any) => v.keyword("logic").signing("signed").dimensions([
        (d: any) => d.rangedimension((r: any) => r.left(SV.literal(7n)).right(SV.literal(0n))), (d: any) => d.sizedimension((z: any) => z.size(SV.literal(4n))),
        (d: any) => d.unsizeddimension((z: any) => z), (d: any) => d.associativedimension((a: any) => a.type(svAtom("int"))),
        (d: any) => d.associativedimension((a: any) => a), (d: any) => d.queuedimension((q: any) => q),
        (d: any) => d.queuedimension((q: any) => q.bound(SV.literal(8n)))]))).declarators([(d: any) => d.name(svIdentifier("a"))]),
    (m: any) => m.attributes([(a: any) => a.specs([(s: any) => s.name(svIdentifier("keep"))])]).type(svAtom("int")).declarators([
        (d: any) => d.name(svIdentifier("b")).dimensions([(x: any) => x.unsizeddimension((y: any) => y)]).value(SV.literal(1n))]),
    (m: any) => m.type((t: any) => t.nonintegertype((n: any) => n.keyword("real"))).declarators([(d: any) => d.name(svIdentifier("c"))]),
    (m: any) => m.type((t: any) => t.keywordtype((k: any) => k.keyword("string"))).declarators([(d: any) => d.name(svIdentifier("d"))]),
    (m: any) => m.type((t: any) => t.namedtype((n: any) => n.name((x: any) => x.identifier(svIdentifier("word_t"))))).declarators([(d: any) => d.name(svIdentifier("e"))]),
    (m: any) => m.type((t: any) => t.virtualinterfacetype((v: any) => v.interface(svIdentifier("bus")).parameters([SV.literal(8n)]).modport(svIdentifier("mp")))).declarators(
        [(d: any) => d.name(svIdentifier("f"))]),
    (m: any) => m.type((t: any) => t.implicittype((i: any) => i.signing("signed"))).declarators([(d: any) => d.name(svIdentifier("g"))]),
    (m: any) => m.type((t: any) => t.enumtype((e: any) => e.base(svAtom("int")).members([(n: any) => n.name(svIdentifier("A")),
        (n: any) => n.name(svIdentifier("B")).left(SV.literal(2n)).value(SV.literal(1n)), (n: any) => n.name(svIdentifier("C")).left(SV.literal(0n)).right(SV.literal(1n))]))).declarators(
        [(d: any) => d.name(svIdentifier("h"))]),
    (m: any) => m.type((t: any) => t.typereference((r: any) => r.operand(svPerson))).declarators([(d: any) => d.name(svIdentifier("i"))])]));
const patternsEvery = (u: any) => u.structurepattern((s: any) => s.items([ // '{a: .x, b: tagged Valid (.*), c: person.a}
    (i: any) => i.patternmember((m: any) => m.name(svIdentifier("a")).pattern((p: any) => p.variablepattern((v: any) => v.name(svIdentifier("x"))))),
    (i: any) => i.patternmember((m: any) => m.name(svIdentifier("b")).pattern((p: any) => p.taggedpattern((t: any) => t.member(svIdentifier("Valid")).pattern(
        (q: any) => q.parenthesizedpattern((r: any) => r.pattern((w: any) => w.wildcardpattern((x: any) => x))))))),
    (i: any) => i.patternmember((m: any) => m.name(svIdentifier("c")).pattern(svMember))]));
const assertionsEvery = (u: any) => u.propertyspec((s: any) => s.clock((k: any) => k.events([
    (e: any) => e.edge("posedge").expression(svPerson).condition(svPerson),
    (e: any) => e.edge("negedge").expression(svPerson)])).disable(svPerson).property((p: any) => p.binaryproperty((o: any) => o.left(
    (l: any) => l.binaryproperty((q: any) => q.left(
        (i: any) => i.implicationproperty((m: any) => m.antecedent((s1: any) => s1.delaysequence((ds: any) => ds.first(
            (ps: any) => ps.parenthesizedsequence((pp: any) => pp.sequence((r: any) => r.repetitionsequence((rr: any) => rr.sequence(svPerson).operator("*").count(
                (cr: any) => cr.cyclerange((c: any) => c.low(SV.literal(1n)).high((dd: any) => dd.dollarexpression((z: any) => z)))))).items([
                (it: any) => it.assignmentexpression((ae: any) => ae.target(svPerson).operator("+=").value(SV.literal(1n)))]))).steps([
            (st: any) => st.delay((cr: any) => cr.cyclerange((c: any) => c.low(SV.literal(1n)).high(SV.literal(3n)))).sequence(
                (fm: any) => fm.firstmatchsequence((f: any) => f.sequence((bs: any) => bs.binarysequence((b2: any) => b2.left(svPerson).operator("intersect").right(svPerson)))))]))).operator("|=>").consequent(
            (cs: any) => cs.clockedsequence((k: any) => k.clock((e: any) => e).sequence((ds: any) => ds.delaysequence((q2: any) => q2.first(svPerson).steps([
                (st: any) => st.delay(SV.literal(1n)).sequence((r: any) => r.repetitionsequence((rr: any) => rr.sequence(svPerson).operator("->").count(SV.literal(2n)))),
                (st: any) => st.delay(svPerson).sequence((r: any) => r.repetitionsequence((rr: any) => rr.sequence(svPerson).operator("=").count(SV.literal(1n))))]))))))).operator("and").right(
        (cp: any) => cp.conditionalproperty((c: any) => c.condition(svPerson).consequence(
            (sp: any) => sp.strengthproperty((w: any) => w.keyword("weak").sequence((bs: any) => bs.binarysequence((b2: any) => b2.left(svPerson).operator("or").right(svPerson))))).alternative(
            (ab: any) => ab.abortproperty((r: any) => r.keyword("reject_on").condition(svPerson).operand((cp2: any) => cp2.caseproperty((cc: any) => cc.expression(svPerson).items([
                (it: any) => it.expressions([SV.literal(1n), SV.literal(2n)]).body((bp: any) => bp.binaryproperty((u2: any) => u2.left(svPerson).operator("until").right(svPerson))),
                (it: any) => it.body((up: any) => up.unaryproperty((u3: any) => u3.operator("not").operand(svPerson)))]))))))))).operator("or").right(
    (pp: any) => pp.parenthesizedproperty((q: any) => q.property((cl: any) => cl.clockedproperty((k: any) => k.clock((e: any) => e.events([
        (ev: any) => ev.edge("negedge").expression(svPerson)])).property((bp: any) => bp.binaryproperty((o2: any) => o2.left(
            (bs: any) => bs.binarysequence((t: any) => t.left(svPerson).operator("throughout").right(svPerson))).operator("iff").right(
            (ip: any) => ip.implicationproperty((im: any) => im.antecedent((up: any) => up.unaryproperty((u4: any) => u4.operator("nexttime").range(SV.literal(2n)).operand(svPerson))).operator("#-#").consequent(svPerson))))))))))));
const svHolds = (e: any) => (u: any) => u.expressionconstraint((c: any) => c.expression(e)); // a constraint of an expression
const constraintsEvery = (u: any) => u.constraintblock((b: any) => b.items([
    (i: any) => i.expressionconstraint((c: any) => c.soft(true).expression((d: any) => d.distexpression((e: any) => e.value(svPerson).items([
        (t: any) => t.value(SV.literal(0n)).operator(":=").weight(SV.literal(40n)),
        (t: any) => t.value(SV.span(1n, 3n)).operator(":/").weight(SV.literal(60n)),
        (t: any) => t.operator(":/").weight(SV.literal(1n))])))), // soft person dist {0 := 40, [1:3] :/ 60, default :/ 1};
    (i: any) => i.implicationconstraint((c: any) => c.condition(SV.binary(">", svPerson, 0n)).body(svHolds(svMember))),
    (i: any) => i.conditionalconstraint((c: any) => c.condition(svPerson).consequence(svHolds(svMember)).alternative(
        (e: any) => e.constraintblock((b: any) => b.items([])))),
    (i: any) => i.foreachconstraint((f: any) => f.value(svPerson).variables([(v: any) => v.identifier(svIdentifier("k")),
                                                                            (v: any) => v.emptyargument((e: any) => e)])
        .body(svHolds(SV.binary("<", SV.identifier("k"), svMember)))),
    (i: any) => i.solvebeforeconstraint((s: any) => s.solve([svMember]).before([svPerson])),
    (i: any) => i.disablesoftconstraint((d: any) => d.target(svPerson)),
    (i: any) => i.uniqueconstraint((q: any) => q.set([svPerson, SV.span(1n, 2n)])),
]));
const preprocessorEvery = (u: any) => u.ifdefdirective((d: any) => d // `ifdef (A && !B) ... `elsif C ... `else ... `endif
    .condition((c: any) => c.parenthesizedexpression((p: any) => p.expression(SV.binary("&&", SV.identifier("A"), SV.unary("!", SV.identifier("B"))))))
    .items([
        (i: any) => i.includedirective((r: any) => r.path("defs.svh")),
        (i: any) => i.includedirective((r: any) => r.path("uvm.svh").system(true)),
        (i: any) => i.definedirective((r: any) => r.name(svIdentifier("MAX")).function_like(true).parameters(
            [svIdentifier("a"), svIdentifier("b")]).body("((a) > (b) ? (a) : (b))")),
        (i: any) => i.undefdirective((r: any) => r.name(svIdentifier("W"))),
        (i: any) => i.timescaledirective((r: any) => r.unit("1ns").precision("1ps")),
        (i: any) => i.defaultnettypedirective((r: any) => r.net_type("none")),
        (i: any) => i.comment((r: any) => r.text(" low")),
        (i: any) => i.comment((r: any) => r.text(" block ").block(true)),
        (i: any) => i.macrousage((r: any) => r.name(svIdentifier("MAX")).arguments("1, 2")),
    ])
    .branches([(b: any) => b.name(svIdentifier("C")).items([(i: any) => i.otherdirective((r: any) => r.text("`resetall"))])])
    .alternative([(i: any) => i.disabledtext((r: any) => r.text("skipped;"))]).has_else(true));
// TypeScript Types: every kind, the terms a type holds among them --- the text:
// any | unknown | number | bigint | boolean | string | symbol | object | never | void | undefined | null | intrinsic | this |
// A.B<T> | (T) | "s" | `a${T}b` | T[] | [a: T, T?, ...T[]] | (T & U) | (<T>(x: T) => x is T) | (abstract new () => T) |
// { a: T; m(x): T; (): T; new (): T; [k: T]: T; // c
// } | { [K in T as U]: K } | T[keyof T] | typeof this | import("m").A | (T extends (infer V)[] ? V : never)
const tsT = (t: any) => t.tstypereference((r: any) => r.typeName((n: any) => n.identifier((i: any) => i.name("T")))); // `T`
const tsAnnotated = (name: string) => (p: any) => p.identifier((i: any) => i.name(name).typeAnnotation((a: any) => a.typeAnnotation(tsT)));
const tsTypesEvery = (u: any) => u.tsuniontype((n: any) => n.types([ // every kind, written through the builders
  ...TST.KEYWORDS.map((keyword) => (k: any) => k[keyword]((h: any) => h)),
  (k: any) => k.tstypereference((r: any) => r.typeName((m: any) => m.tsqualifiedname((q: any) => q.left((l: any) => l.identifier(
    (i: any) => i.name("A"))).right((i: any) => i.name("B")))).typeArguments((a: any) => a.params([tsT]))),
  (k: any) => k.tsparenthesizedtype((p: any) => p.typeAnnotation(tsT)),
  (k: any) => k.tsliteraltype((p: any) => p.literal((li: any) => li.literal((v: any) => v.value((x: any) => x.as_scalar("s"))))),
  (k: any) => k.tstemplateliteraltype((p: any) => p.quasis([(q: any) => q.raw("a"), (q: any) => q.raw("b").tail(true)]).types([tsT])),
  (k: any) => k.tsarraytype((a: any) => a.elementType(tsT)),
  (k: any) => k.tstupletype((p: any) => p.elementTypes([
    (e: any) => e.tsnamedtuplemember((m: any) => m.label((i: any) => i.name("a")).elementType(tsT)),
    (e: any) => e.tsoptionaltype((o: any) => o.typeAnnotation(tsT)),
    (e: any) => e.tsresttype((o: any) => o.typeAnnotation((a: any) => a.tsarraytype((b: any) => b.elementType(tsT))))])),
  (k: any) => k.tsintersectiontype((p: any) => p.types([tsT, (t: any) => t.tstypereference((r: any) => r.typeName(
    (m: any) => m.identifier((i: any) => i.name("U"))))])),
  (k: any) => k.tsfunctiontype((f: any) => f.typeParameters((d: any) => d.params([(p: any) => p.name((i: any) => i.name("T"))])).params([
    tsAnnotated("x")]).returnType((a: any) => a.typeAnnotation((p: any) => p.tstypepredicate((q: any) => q.parameterName(
      (m: any) => m.identifier((i: any) => i.name("x"))).typeAnnotation((b: any) => b.typeAnnotation(tsT)))))),
  (k: any) => k.tsconstructortype((f: any) => f.abstract(true).returnType((a: any) => a.typeAnnotation(tsT))),
  (k: any) => k.tstypeliteral((q: any) => q.members([
    (m: any) => m.tspropertysignature((s: any) => s.key((y: any) => y.identifier((i: any) => i.name("a"))).typeAnnotation((a: any) => a.typeAnnotation(tsT))),
    (m: any) => m.tsmethodsignature((s: any) => s.key((y: any) => y.identifier((i: any) => i.name("m"))).params([
      (p: any) => p.identifier((i: any) => i.name("x"))]).returnType((a: any) => a.typeAnnotation(tsT))),
    (m: any) => m.tscallsignaturedeclaration((s: any) => s.returnType((a: any) => a.typeAnnotation(tsT))),
    (m: any) => m.tsconstructsignaturedeclaration((s: any) => s.returnType((a: any) => a.typeAnnotation(tsT))),
    (m: any) => m.tsindexsignature((s: any) => s.parameters([tsAnnotated("k")]).typeAnnotation((a: any) => a.typeAnnotation(tsT))),
    (m: any) => m.comment((c: any) => c.text(" c"))])),
  (k: any) => k.tsmappedtype((m: any) => m.key((i: any) => i.name("K")).value(tsT).nameType((t: any) => t.tstypereference(
    (r: any) => r.typeName((n: any) => n.identifier((i: any) => i.name("U"))))).body((t: any) => t.tstypereference(
    (r: any) => r.typeName((n: any) => n.identifier((i: any) => i.name("K")))))),
  (k: any) => k.tsindexedaccesstype((a: any) => a.objectType(tsT).indexType((i: any) => i.tstypeoperator(
    (o: any) => o.operator("keyof").typeAnnotation(tsT)))),
  (k: any) => k.tstypequery((q: any) => q.exprName((e: any) => e.thisexpression((h: any) => h))),
  (k: any) => k.tsimporttype((i: any) => i.source((s: any) => s.literal((li: any) => li.value((x: any) => x.as_scalar("m")))).qualifier(
    (q: any) => q.identifier((j: any) => j.name("A")))),
  (k: any) => k.tsconditionaltype((c: any) => c.checkType(tsT).extendsType((e: any) => e.tsarraytype((a: any) => a.elementType(
    (i: any) => i.tsinfertype((f: any) => f.typeParameter((p: any) => p.name((n: any) => n.name("V"))))))).trueType(
    (t: any) => t.tstypereference((r: any) => r.typeName((n: any) => n.identifier((i: any) => i.name("V"))))).falseType(
    (t: any) => t.tsneverkeyword((h: any) => h))),
]));
// TypeScript Patterns: every kind, the terms a target may be among them --- the text:
// [{ a: person = 1, ...person }, , person.x, private person]
const tsPatternsEvery = (u: any) => u.arraypattern((a: any) => a.elements([ // every kind, written through the builders
  (e: any) => e.objectpattern((o: any) => o.properties([
    (p: any) => p.property((q: any) => q.propertyKind("init").key((k: any) => k.identifier((i: any) => i.name("a"))).value(
      (v: any) => v.assignmentpattern((s: any) => s.left(tsPerson).right((r: any) => r.literal((li: any) => li.value(
        (x: any) => x.as_scalar(1.0))))))),
    (p: any) => p.restelement((r: any) => r.argument(tsPerson))])),
  (e: any) => e.elision((h: any) => h),
  (e: any) => e.memberexpression((m: any) => m.object(tsPerson).property((y: any) => y.identifier((i: any) => i.name("x")))),
  (e: any) => e.tsparameterproperty((t: any) => t.accessibility("private").parameter(tsPerson))]));
// The regular expression dialects, each of every kind, their properties named as regexpp names ECMAScript's.
// Python RegExp --- the text: (?:a|)(?P<n>[a-z\d]+?)(?<!b)(?P=n)(?(1)a)(?>.)b*+^(?#c)|(?i-m:a)
const pyrEvery = (u: any) => u.pattern((p: any) => p.alternatives([
  (s: any) => s.alternative((a: any) => a.elements([
    (e: any) => e.group((g: any) => g.alternatives([
      (x: any) => x.alternative((y: any) => y.elements([(c: any) => c.character((v: any) => v.value(97n))])),
      (x: any) => x.alternative((y: any) => y)])),
    (e: any) => e.capturinggroup((g: any) => g.name("n").alternatives([(x: any) => x.alternative((y: any) => y.elements([
      (q: any) => q.quantifier((r: any) => r.min(1n).greedy(false).element((k: any) => k.characterclass((c: any) => c.elements([
        (r: any) => r.characterclassrange((n: any) => n.min((c: any) => c.character((v: any) => v.value(97n))).max(
          (c: any) => c.character((v: any) => v.value(122n)))),
        (s: any) => s.characterset((v: any) => v.setKind("digit"))]))))]))])),
    (e: any) => e.assertion((t: any) => t.assertionKind("lookbehind").negate(true).alternatives([
      (x: any) => x.alternative((y: any) => y.elements([(c: any) => c.character((v: any) => v.value(98n))]))])),
    (e: any) => e.backreference((r: any) => r.ref("n")),
    (e: any) => e.conditional((c: any) => c.ref(1n).yes((x: any) => x.alternative((y: any) => y.elements([
      (c: any) => c.character((v: any) => v.value(97n))])))),
    (e: any) => e.atomicgroup((g: any) => g.alternatives([(x: any) => x.alternative((y: any) => y.elements([
      (s: any) => s.characterset((v: any) => v.setKind("any"))]))])),
    (e: any) => e.quantifier((q: any) => q.min(0n).possessive(true).element((c: any) => c.character((v: any) => v.value(98n)))),
    (e: any) => e.assertion((t: any) => t.assertionKind("start")),
    (e: any) => e.comment((c: any) => c.text("c"))])),
  (s: any) => s.alternative((a: any) => a.elements([
    (e: any) => e.flags((f: any) => f.add("i").remove("m").alternatives([(x: any) => x.alternative((y: any) => y.elements([
      (c: any) => c.character((v: any) => v.value(97n))]))]))]))]));
// TypeScript RegExp --- the text: /(?i:a)(?<n>b{1,}?)[a-c\d][\q{x|y}&&[x]][\w--a]\p{L}\b(?=c)\k<n>|$/v
const tsChar = (c: string) => (e: any) => e.character((h: any) => h.value(BigInt(c.codePointAt(0)!)));
const tsSeq = (...items: unknown[]) => (e: any) => e.alternative((a: any) => a.elements(items));
const tsrEvery = (u: any) => u.regexpliteral((l: any) => l.pattern((p: any) => p.pattern((q: any) => q.alternatives([
  tsSeq((e: any) => e.group((g: any) => g.modifiers((m: any) => m.modifiers((s: any) => s.add((f: any) => f.ignoreCase(true)))).alternatives(
    [tsSeq(tsChar("a"))])),
  (e: any) => e.capturinggroup((g: any) => g.name("n").alternatives([tsSeq((e: any) => e.quantifier((q: any) => q.raw("b{1,}?").min(1n).greedy(
    false).element(tsChar("b"))))])),
  (e: any) => e.characterclass((c: any) => c.elements([(e: any) => e.characterclassrange((r: any) => r.min(tsChar("a")).max(tsChar("c"))),
    (e: any) => e.characterset((s: any) => s.setKind("digit"))])),
  (e: any) => e.expressioncharacterclass((x: any) => x.expression((e: any) => e.classintersection((i: any) => i.left(
    (e: any) => e.classstringdisjunction((d: any) => d.alternatives([(e: any) => e.stringalternative((s: any) => s.elements([tsChar("x")])),
      (e: any) => e.stringalternative((s: any) => s.elements([tsChar("y")]))]))).right(
    (e: any) => e.characterclass((c: any) => c.elements([tsChar("x")])))))),
  (e: any) => e.expressioncharacterclass((x: any) => x.expression((e: any) => e.classsubtraction((s: any) => s.left(
    (e: any) => e.characterset((t: any) => t.setKind("word"))).right(tsChar("a"))))),
  (e: any) => e.characterset((s: any) => s.setKind("property").key("L")), (e: any) => e.assertion((a: any) => a.assertionKind("word")),
  (e: any) => e.assertion((a: any) => a.assertionKind("lookahead").alternatives([tsSeq(tsChar("c"))])),
  (e: any) => e.backreference((b: any) => b.ref("n"))),
  tsSeq((e: any) => e.assertion((a: any) => a.assertionKind("end")))]))).flags((f: any) => f.unicodeSets(true)));
// Cpp RegExp --- ECMAScript's: ^(a|.)[^a-[.ch.][:alpha:][=e=]\d]{1,3}.$|(?:b)(?!c)x*?\1\b
const regexpChar = (c: string) => (u: any) => u.character((r: any) => r.value(BigInt(c.codePointAt(0) as number))); // a character, of every RegExp dialect here
const regexpAlt = (...elements: unknown[]) => (u: any) => u.alternative((r: any) => r.elements(elements));
const xrEvery = (u: any) => u.regex((r: any) => r.flags(["ECMAScript"]).pattern(XR.spec((p: any) => p.pattern((w: any) => w.alternatives([
  regexpAlt(
    (e: any) => e.assertion((a: any) => a.assertionKind("start")),
    (e: any) => e.capturinggroup((g: any) => g.alternatives([regexpAlt(regexpChar("a")), regexpAlt(regexpChar("."))])),
    (e: any) => e.quantifier((m: any) => m.min(1n).max(3n).element((c: any) => c.characterclass((k: any) => k.negate(true).elements([
      (x: any) => x.characterclassrange((y: any) => y.min(regexpChar("a")).max((z: any) => z.collatingelement((n: any) => n.name("ch")))),
      (x: any) => x.exclass((n: any) => n.name("alpha")), (x: any) => x.equivalence((n: any) => n.name("e")),
      (x: any) => x.characterset((n: any) => n.setKind("digit"))])))),
    (e: any) => e.characterset((n: any) => n.setKind("any")), (e: any) => e.assertion((a: any) => a.assertionKind("end"))),
  regexpAlt(
    (e: any) => e.group((g: any) => g.alternatives([regexpAlt(regexpChar("b"))])),
    (e: any) => e.assertion((a: any) => a.assertionKind("lookahead").negate(true).alternatives([regexpAlt(regexpChar("c"))])),
    (e: any) => e.quantifier((m: any) => m.raw("*?").min(0n).greedy(false).element(regexpChar("x"))),
    (e: any) => e.backreference((b: any) => b.ref(1n)), (e: any) => e.assertion((a: any) => a.assertionKind("word")))])))));
// POSIX BRE --- grep's list of two patterns: ^\(a\)[^a-[.ch.][:alpha:][=e=]]\{1,3\}.\1$, then b*
const posixClass = (c: any) => c.characterclass((k: any) => k.negate(true).elements([ // [^a-[.ch.][:alpha:][=e=]]
  (x: any) => x.characterclassrange((y: any) => y.min(regexpChar("a")).max((z: any) => z.collatingelement((n: any) => n.name("ch")))),
  (x: any) => x.exclass((n: any) => n.name("alpha")), (x: any) => x.equivalence((n: any) => n.name("e"))]));
const breEvery = (u: any) => u.patternlist((q: any) => q.patterns([
  (v: any) => v.pattern((w: any) => w.alternatives([regexpAlt(
    (e: any) => e.assertion((a: any) => a.assertionKind("start")),
    (e: any) => e.capturinggroup((g: any) => g.alternatives([regexpAlt(regexpChar("a"))])),
    (e: any) => e.quantifier((m: any) => m.min(1n).max(3n).element(posixClass)),
    (e: any) => e.characterset((n: any) => n.setKind("any")), (e: any) => e.backreference((b: any) => b.ref(1n)),
    (e: any) => e.assertion((a: any) => a.assertionKind("end")))])),
  (v: any) => v.pattern((w: any) => w.alternatives([regexpAlt((e: any) => e.quantifier((m: any) => m.min(0n).element(regexpChar("b"))))]))]));
// POSIX ERE --- grep -E's list of two patterns: ^(a|.)[^a-[.ch.][:alpha:][=e=]]{1,3}?$, then b+{0,1}
const ereEvery = (u: any) => u.patternlist((q: any) => q.patterns([
  (v: any) => v.pattern((w: any) => w.alternatives([regexpAlt(
    (e: any) => e.assertion((a: any) => a.assertionKind("start")),
    (e: any) => e.capturinggroup((g: any) => g.alternatives([regexpAlt(regexpChar("a")),
      regexpAlt((s: any) => s.characterset((n: any) => n.setKind("any")))])),
    (e: any) => e.quantifier((m: any) => m.min(1n).max(3n).greedy(false).element(posixClass)),
    (e: any) => e.assertion((a: any) => a.assertionKind("end")))])),
  (v: any) => v.pattern((w: any) => w.alternatives([regexpAlt((e: any) => e.quantifier((m: any) => m.min(0n).max(1n).element(
    (f: any) => f.quantifier((n: any) => n.min(1n).element(regexpChar("b"))))))]))]));
const samples: Record<string, any> = { // one term per dialect, with every kind
  Patterns: term(E.operation("and", basic, E.list_(1n, E.mapping(["k", person.items]).item("k")).all( // a quantifier too
    "x", E.variable("x").gt(0n)))),
  Python3: term(P.import_("numpy", P.importfrom("math", "floor", P.let_("a", P.name("person"), P.ifexp( // every kind
    P.boolop("or", P.boolop("and", P.unaryop("not", P.compare("==", P.attribute(P.name("a"), "z"), 1n, ["<", 2n])),
      P.unaryop("-", P.attribute(P.name("a"), "w"))), P.binop("*", 2n, 3n), P.namedexpr("q", 0n)),
    P.call("np.add", P.attribute(P.name("a"), "x"), P.subscript(P.name("a"), P.tuple_(P.slice_(1n, null, -1n), P.constant(P.ELLIPSIS))),
      P.starred(P.list_(1n)), P.keyword("k", P.dict_(["k", 2n], [null, P.name("a")]))),
    P.call("f", P.generatorexp(P.subscript(P.name("p"), 0n), P.comprehension("p", P.attribute(P.name("a"), "z"), [P.name("p")])),
      P.listcomp(P.name("p"), P.comprehension(P.tuple_(P.name("p"), P.starred(P.name("r"))), P.name("a"), [], true)),
      P.setcomp(P.name("p"), P.comprehension("p", P.name("a"))), P.dictcomp(P.name("p"), null, P.comprehension("p", P.name("a"))),
      P.set_(P.constant(null), P.constant(new P.Complex(0, 1))), P.lambda_(P.name("b"), [P.arg("b", 1n)]), P.yield_(),
      P.yieldfrom(P.name("a")), P.await_(P.name("a")), P.concatenatedstring("x", P.joinedstr(["y", P.formattedvalue(P.name("a"), "r")])),
      P.templatestr([P.interpolation(P.name("a"))]), P.parenthesized(P.name("a"))))),
  "f"), "np"), P.DIALECT),
  MATLAB: term(M.import_("pkg.*", M.binary("&&", M.unary("~", M.field(M.identifier("person"), "age")),
    M.call("isfield", "x", M.index(M.array(M.arrayfun("p", M.identifier("person"), M.identifier("p")), 2n), 1n)))), M.DIALECT),
  Excel: term(X.let_("a", X.field(X.name("person"), "age"), X.function("AND", X.infix(">=", X.name("a"), X.cell("A1")),
    X.prefix("-", X.function("SUM", X.map_("p", X.name("a"), X.infix("*", X.name("p"), 2n)))))), X.DIALECT),
  LaTeX: term(L.where("a", L.member(L.symbol("person"), "age"), L.binary("\\land", L.binary("\\geq", L.symbol("a"), 18n), L.binary(
    "\\lor", L.unary("\\lnot", L.function("has", L.symbol("person"), "email")), L.binary(">", L.frac(L.symbol("a"), 2n), 1.5)))), L.DIALECT),
  C: term(cEvery, C.DIALECT),
  Cpp: term(cppEvery, CPP.DIALECT),
  "C Types": term(enumeration(CT, C), CT),
  "Cpp Types": term(enumeration(XT, CPP), XT),
  "C Preprocessor": term(cPreprocessorEvery, CPRE),
  SystemVerilog: term(svEvery, SV.DIALECT),
  "SystemVerilog Types": term(typesEvery, SVT.DIALECT),
  "SystemVerilog Patterns": term(patternsEvery, SVP.DIALECT),
  "SystemVerilog Assertions": term(assertionsEvery, SVA.DIALECT),
  "SystemVerilog Constraints": term(constraintsEvery, SVC.DIALECT),
  "SystemVerilog Preprocessor": term(preprocessorEvery, SVPP.DIALECT),
  TypeScript5: term(typescript5, TS.DIALECT),
  "TypeScript Types": term(tsTypesEvery, TST.DIALECT),
  "TypeScript Patterns": term(tsPatternsEvery, TSP.DIALECT),
  "Python Types": term((u: any) => u.typevar((t: any) => t.name("T").bound( // T: **P = [int, person] = *Ts = *person
    (b: any) => b.paramspec((s: any) => s.name("P").default_value(P.list_(P.name("int"), P.name("person"))))).default_value(
    (d: any) => d.typevartuple((s: any) => s.name("Ts").default_value(P.starred(P.name("person")))))), PYT.DIALECT),
  "Python Patterns": term((u: any) => u.matchor((o: any) => o.patterns([ // person.Point(0, y=[a, *rest]) | {'k': None, **kw} | (person.RED as c)
    (p: any) => p.matchclass((c: any) => c.cls(P.attribute(P.name("person"), "Point")).patterns([
      (v: any) => v.matchvalue((m: any) => m.value(P.constant(0n)))]).kwd_attrs(["y"]).kwd_patterns([
      (s: any) => s.matchsequence((q: any) => q.delimiters("[]").patterns([(a: any) => a.matchas((m: any) => m.name("a")),
        (r: any) => r.matchstar((m: any) => m.name("rest"))]))])),
    (p: any) => p.matchmapping((m: any) => m.keys([P.constant("k")]).patterns([(s: any) => s.matchsingleton((v: any) => v.value("None"))]).rest("kw")),
    (p: any) => p.matchas((m: any) => m.pattern((v: any) => v.matchvalue((w: any) => w.value(P.attribute(P.name("person"), "RED")))).name("c"))])),
    PYP.DIALECT),
  "Python RegExp": term(pyrEvery, PYR.DIALECT),
  "TypeScript RegExp": term(tsrEvery, TSR.DIALECT),
  "Cpp RegExp": term(xrEvery, XR),
  "POSIX BRE": term(breEvery, POSIX[0]),
  "POSIX ERE": term(ereEvery, POSIX[1]),
};
const DIALECTS = [E.DIALECT, P.DIALECT, M.DIALECT, X.DIALECT, L.DIALECT, C.DIALECT, CPP.DIALECT, SV.DIALECT, SVT.DIALECT,
  SVP.DIALECT, SVA.DIALECT, SVC.DIALECT, SVPP.DIALECT, TS.DIALECT, TST.DIALECT, TSP.DIALECT, PYT.DIALECT, PYP.DIALECT,
  PYR.DIALECT, TSR.DIALECT];
const FAMILY = DIALECTS.slice(7, 13); // SystemVerilog's six dialects, each reading the others' terms
const TS_FAMILY = [...DIALECTS.slice(13, 16), TSR.DIALECT]; // TypeScript's four, each reading the others' terms
const PYTHON = [P.DIALECT, PYT.DIALECT, PYP.DIALECT, PYR.DIALECT]; // Python's four, each reading the others' terms
const READER = new Map([...DIALECTS, ...C_FAMILY, ...POSIX].map((d) => [d.union, d])); // the dialect whose union holds a term
const SIMPLIFIERS = new Map<F.Dialect, FV.Simplifier>([[E.DIALECT, V.SIMPLIFIER], [P.DIALECT, Python3.Simplifiers.SIMPLIFIER],
  [M.DIALECT, Matlab.Simplifiers.SIMPLIFIER], [X.DIALECT, Excel.Simplifiers.SIMPLIFIER], [C.DIALECT, CDialect.Simplifiers.SIMPLIFIER],
  [CPP.DIALECT, CppDialect.Simplifiers.SIMPLIFIER], [SV.DIALECT, SystemVerilog.Simplifiers.SIMPLIFIER],
  [TS.DIALECT, TypeScript5.Simplifiers.SIMPLIFIER]]); // LaTeX has none
const INVALID = new Set(["error", "missing"]); // the framework's kinds of invalid syntax, which every dialect's `Term` holds

<a id="FRM-01"></a>
## FRM-01 · Every dialect, expression, evaluator and translator implements its protocol

In [ ]:
assert([...SIMPLIFIERS.values()].every((simplify) => simplify instanceof FV.Simplifier));
for (const dialect of [...DIALECTS, ...C_FAMILY, ...POSIX]) {
  assert(dialect instanceof F.Dialect && [...dialect.kinds().values()].every((kind) => kind instanceof F.Kind));
  for (const node of F.walk(samples[dialect.name()])) { // a term of its own union, nested in its kinds or in shared ones
    if (dialect.reads().length > 0) { // or a term of a dialect of its family, which it reads
      const reader = READER.get(F.dialectOf(node)) as F.Dialect;
      assert(F.isTerm(node) && [dialect, ...dialect.reads()].includes(reader) && dialect.kind(node)[0] === F.kindOf(node)[0]);
      continue;
    }
    assert(F.isTerm(node) && F.dialectOf(node) === dialect.union && dialect.kinds().has(F.kindOf(node)[0]));
  }
}
assert(same(DIALECTS.map((d) => d.name()), ["Patterns", "Python3", "MATLAB", "Excel", "LaTeX", "C", "Cpp", "SystemVerilog",
  "SystemVerilog Types", "SystemVerilog Patterns", "SystemVerilog Assertions", "SystemVerilog Constraints",
  "SystemVerilog Preprocessor", "TypeScript5", "TypeScript Types", "TypeScript Patterns", "Python Types", "Python Patterns",
  "Python RegExp", "TypeScript RegExp"]));
// A sublanguage's dialect is named by its family and its own name; its tag, its name lowercased, a space an underscore.
assert(same(FAMILY.map((d) => d.tag), ["systemverilog", "systemverilog_types", "systemverilog_patterns",
  "systemverilog_assertions", "systemverilog_constraints", "systemverilog_preprocessor"]));
assert(same(["SystemVerilog.Types", "C.RegExp", "Python.Python3", "C.Expressions", "POSIX.BRE"].map((path) => F.dialectName(path)), [
  "SystemVerilog Types", "C RegExp", "Python3", "C", "POSIX BRE"]) && F.dialectTag("SystemVerilog Types") === "systemverilog_types");
assert(same(F.SUBLANGUAGES, ["Types", "Patterns", "Assertions", "Constraints", "Preprocessor", "RegExp", "BRE", "ERE"]));
for (const family of [FAMILY, TS_FAMILY, PYTHON]) { // each reads the others' terms, in its family's order
  for (const dialect of family) {
    assert(same(dialect.reads(), family.filter((d) => d !== dialect)) && dialect.reads() === dialect.reads());
  }
}
assert(same(TS_FAMILY.map((d) => d.tag), ["typescript5", "typescript_types", "typescript_patterns", "typescript_regexp"]));
// The retrofit's new dialects: each a package exporting its install and define, its singleton, its `Term`, its holder,
// its printer and its `reads`, each family's dialects reading one another.
const NEW: F.Dialect[] = RETROFIT.map((pkg) => pkg.Expressions.DIALECT);
const [PT, PP, TT, TP] = NEW as [F.Dialect, F.Dialect, F.Dialect, F.Dialect];
const [PR, TR] = NEW.slice(7, 9) as [F.Dialect, F.Dialect]; // the regular expression dialects, installed last: Python's,
// TypeScript's, C++'s, XR, and POSIX's
assert(same([...NEW.slice(4, 7), ...NEW.slice(9, 10)], C_FAMILY) && same(NEW.slice(10), POSIX)); // the C family's: C Types,
// Cpp Types, C Preprocessor, Cpp RegExp
assert(same(NEW.map((d) => d.name()), ["Python Types", "Python Patterns", "TypeScript Types", "TypeScript Patterns", "C Types",
  "Cpp Types", "C Preprocessor", "Python RegExp", "TypeScript RegExp", "Cpp RegExp", "POSIX BRE", "POSIX ERE"]));
assert(same(NEW.map((d) => d.tag), ["python_types", "python_patterns", "typescript_types", "typescript_patterns", "c_types",
  "cpp_types", "c_preprocessor", "python_regexp", "typescript_regexp", "cpp_regexp", "posix_bre", "posix_ere"]));
const READS = new Map<F.Dialect, F.Dialect[]>([[P.DIALECT, [PT, PP, PR]], [PT, [P.DIALECT, PP, PR]], [PP, [P.DIALECT, PT, PR]],
  [PR, [P.DIALECT, PT, PP]], [TS.DIALECT, [TT, TP, TR]], [TT, [TS.DIALECT, TP, TR]], [TP, [TS.DIALECT, TT, TR]],
  [TR, [TS.DIALECT, TT, TP]], [C.DIALECT, [CT, CPRE]], [CT, [C.DIALECT, CPRE]], [CPP.DIALECT, [XT, CPRE, XR]],
  [XT, [CPP.DIALECT, CPRE, XR]], [CPRE, [C.DIALECT, CT, CPP.DIALECT, XT]], [XR, [CPP.DIALECT, XT, CPRE, ...POSIX]],
  [POSIX[0] as F.Dialect, [POSIX[1] as F.Dialect]], [POSIX[1] as F.Dialect, [POSIX[0] as F.Dialect]]]); // C++'s read C's
// preprocessor alone of C's dialects; the preprocessor reads C's and C++'s expressions and types, a condition's and an
// enumeration's; C++'s dialects read one another, its regular expressions among them, as each family's do, and those
// read POSIX's too, which a regex holds; POSIX's read each other's
assert([...DIALECTS, ...NEW].filter((d) => !FAMILY.includes(d)).every((d) => same(d.reads(), READS.get(d) ?? [])));
RETROFIT.forEach((pkg, i) => { // each its package, with its printer, its own kinds, its holder, which names its ABI
  const dialect = NEW[i] as F.Dialect;
  assert(same(Object.keys(pkg).sort(), ["Expressions", "Text", "define", "install"]) && pkg.install === pkg.Expressions.install);
  assert(pkg.define === pkg.Expressions.define && same([...dialect.kinds().keys()],
    [...pkg.Expressions.TAGS, ...(dialect === TT ? ["identifier"] : []), "error", "missing"])); // TypeScript Types' own identifier
  const holder: any = store.registered(`Expressions.${dialect.path}.Expression`);
  assert(holder.properties.get("root").schema === store.registered(dialect.union) && same([...holder.adjacencies.keys()], ["abi"]));
  assert(holder.refines === store.registered("Expressions.Framework.Expression") && pkg.Text.ToText(samples[dialect.name()]) !== "");
});
// A kind's owner, whose name its refusals and its printing of invalid syntax give: the dialect whose `Term` holds it,
// or, for a kind that `Term` lists and a dialect it reads declares, that one; a term of a dialect it does not read, itself.
const typevar = term((u: any) => u.typevar((t: any) => t.name("T")), PT);
const listed = term(TT.make(new F.Form("literal", { value: 1.0 })), TT); // TypeScript5's literal, which Types' `Term` lists
assert(P.DIALECT.owner(typevar) === PT && PT.owner(typevar) === PT && C.DIALECT.owner(typevar) === C.DIALECT);
assert(TT.owner(listed) === TS.DIALECT && TS.DIALECT.owner(listed) === TS.DIALECT);
for (const pkg of [SVTypes, SVPatterns, SVAssertions, SVConstraints, SVPreprocessor] as any[]) { // each its package, its exports alike
  assert(same(Object.keys(pkg).sort(), ["Expressions", "Text", "define", "install"]) && pkg.install === pkg.Expressions.install);
  assert(pkg.define === pkg.Expressions.define && FAMILY.includes(pkg.Expressions.DIALECT));
  const holder = store.registered(`Expressions.${pkg.Expressions.DIALECT.path}.Expression`); // each its holder
  assert(holder.refines === store.registered("Expressions.Framework.Expression") && same([...holder.adjacencies.keys()], ["abi"]));
}
assert(same(Object.keys(SystemVerilog).sort(), ["Domains", "Expressions", "Simplifiers", "Text", "define", "install"]));
const ORDER = [...DIALECTS.slice(0, 14), ...NEW]; // in install order: each new dialect after its family's
assert(Repr.repr(E.DIALECT) === "<dialect Patterns>" && same(ORDER.map((d) => d.tag), Term.branches.map((b: any) => b.name)));
assert(same(ORDER.map((d) => d.union), Term.branches.map((b: any) => b.schema.name))); // the union of the dialects' `Term`s
// Every dialect a branch of the framework's kind union of dialects: the standard dialects first, which have no terms;
// the framework's union of kinds is of the dialects' unions, each holding its dialect's kinds.
const [Dialects, DomainKind]: any[] = [store.registered("Expressions.Framework.Dialect"), store.registered("Expressions.Framework.DomainKind")];
assert(same(Dialects.branches.map((b: any) => b.name), ["isoiec11404", "ieee754", "ieee1164", "unicode", ...DIALECTS.slice(0, 7).map((d) => d.tag),
  "amd64", "aapcs64", ...ORDER.slice(7).map((d) => d.tag)])); // in install order: the ABIs, terms none
assert(same(Dialects.branches.slice(20).map((b: any) => b.schema.singleton), NEW.map((d) => `Expressions.${d.path}`)));
assert(same(Dialects.branches.slice(13, 19).map((b: any) => b.schema.singleton), FAMILY.map((d) => `Expressions.${d.path}`)));
assert(same(Dialects.branches.slice(11, 13).map((b: any) => b.schema.singleton), ["Expressions.SystemV.AMD64", "Expressions.Arm.AAPCS64"]));
assert(same(Dialects.branches.slice(0, 4).map((b: any) => F.dialectName((b.schema.singleton as string).replace(/^Expressions\./, ""))), [
  "ISOIEC11404", "IEEE754", "IEEE1164", "Unicode"]));
assert(equal(new Map(DomainKind.branches.map((b: any) => [b.name, b.schema.branches.map((k: any) => k.name)])), new Map<string, string[]>([
  ["isoiec11404", ["boolean", "integer", "modulo", "bitstring", "octet", "octetstring", "state", "packed"]], ["ieee754", ["ieee754"]],
  ["ieee1164", ["ieee1164"]], ["unicode", ["unicode"]], ["python3", ["str", "nonetype", "ellipsis"]], ["matlab", ["string"]],
  ["excel", ["text", "error"]],
  ["c", ["nullptr_t"]], ["cpp", ["nullptr_t"]], ["systemverilog", ["logic", "bit"]], ["typescript5", ["string", "null"]]])));
assert(same(DomainKind.branches.slice(4).map((b: any) => b.schema.name), DIALECTS // each dialect's own union of kinds, named by its path
  .filter((d) => !["Patterns", "LaTeX"].includes(d.name()) && ![...FAMILY.slice(1), ...TS_FAMILY.slice(1), ...PYTHON.slice(1)].includes(d))
  .map((d) => d.union.replace(/\.Term$/, "") + ".DomainKind")));
assert(Dialects.branches.every((b: any) => same([...b.schema.adjacencies.keys()], ["domains", "operations", "standards"])));
assert(same([...X.DIALECT.kinds().keys()], ["literal", "variable", "cell", "let", "call", "infix", "prefix", "field", "map", "error",
  "missing"]));
const HELD = new Map([[C.DIALECT, ["typeid", "definedexpression"]], [CPP.DIALECT, ["typeid", "definedexpression"]]]); // its
// Types' and the preprocessor's kinds its subterms may be, which C's and C++'s `Term`s list
for (const dialect of DIALECTS) { // each union's branches are its dialect's kinds, named plainly; shared kinds, the framework's
  const union = store.registered(dialect.union);
  if (dialect === PT) { // Python Types', Python3's too: a type expression is any expression
    assert(equal(union.branches.map((b: any) => b.name).sort(), [...new Set([...PT.kinds().keys(), ...P.DIALECT.kinds().keys()])].sort()));
    continue;
  }
  if (FAMILY.includes(dialect)) { // and in SystemVerilog's family, the kinds of the dialects it reads that its subterms may be
    const own = [...dialect.kinds().keys(), ...(dialect !== SVPP.DIALECT ? ["macrousage", ...SV.DIALECT.kinds().keys()] : [])];
    own.push(...(dialect === SVA.DIALECT ? ["distexpression"] : dialect === SVC.DIALECT ? [...SVPP.DIRECTIVES, "comment"] : [])); // held
    assert(equal(union.branches.map((b: any) => b.name).sort(), [...new Set(own)].sort()), dialect.name());
    continue;
  }
  if (TS_FAMILY.includes(dialect)) { // and in TypeScript's, the terms of the dialects it reads that its subterms may be
    const held = dialect === TS.DIALECT ? TSP.TAGS.slice(0, 3) : dialect === TST.DIALECT ? TST.TERMS : dialect === TSP.DIALECT ? TSP.TERMS : [];
    const own = [...dialect.kinds().keys()].filter((tag) => dialect !== TST.DIALECT || tag !== "identifier");
    assert(equal(union.branches.map((b: any) => b.name).sort(), [...new Set([...own, ...held])].sort()), dialect.name());
    continue;
  }
  assert(equal(union.branches.map((b: any) => b.name).sort(), [...dialect.kinds().keys(), ...HELD.get(dialect) ?? []].sort()), dialect.name());
  const shared = union.branches.filter((b: any) => (b.schema.name as string).replace(/\.[^.]*$/, "") === "Expressions.Framework").map((b: any) => (b.schema.name as string).split(".").pop()!.toLowerCase());
  assert(shared.every((tag: string) => ["literal", "call", "variable", "let", "list", "mapping", "error", "missing"].includes(tag))
    && [...INVALID].every((tag) => shared.includes(tag)));
}
const [C_UNION, CPP_UNION] = [store.registered(C.DIALECT.union), store.registered(CPP.DIALECT.union)];
const names = (union: any): string[] => union.branches.map((b: any) => b.name);
assert(same(names(C_UNION), [...[...C.DIALECT.kinds().keys()].slice(0, -2), "typeid", "error", "missing", "definedexpression"]));
assert(same(names(CPP_UNION), [...[...CPP.DIALECT.kinds().keys()].slice(0, -2), "typeid", "error", "missing", "definedexpression"]));
assert(same(names(C_UNION).filter((name) => !names(CPP_UNION).includes(name)), ["variable", "genericselection", "compoundliteralexpression"]));
assert(!CPP.DIALECT.kinds().has("variable")); // C++ writes a name only as an idexpression, its grammar's one kind
assert(same(names(CPP_UNION).filter((name) => !names(C_UNION).includes(name)), [ // C++ is not a superset of C
  "userdefinedliteral", "idexpression", "thisexpression", "lambdaexpression", "foldexpression", "requiresexpression",
  "functionalcastexpression", "namedcastexpression", "typeidexpression", "awaitexpression", "sizeofpackexpression",
  "noexceptexpression", "newexpression", "deleteexpression", "throwexpression", "yieldexpression", "packexpansion"]));
const reused = (b: any) => ["variable", "error", "missing", "typeid", "definedexpression"].includes(b.name);
assert(C_UNION.branches.every((b: any) => reused(b) || b.schema.name.startsWith("Expressions.C.Expressions.")));
assert(CPP_UNION.branches.every((b: any) => ["error", "missing", "typeid", "definedexpression"].includes(b.name)
  || b.schema.name.startsWith("Expressions.Cpp.Expressions.")));
assert(equal(new Map(C_UNION.branches.filter((b: any) => HELD.get(C.DIALECT)!.includes(b.name)).map((b: any) => [b.name, b.schema.name])),
  new Map([["typeid", "Expressions.C.Types.TypeId"], ["definedexpression", "Expressions.C.Preprocessor.DefinedExpression"]]))); // each its own dialect's
assert(same([...E.DIALECT.kinds().keys()], ["literal", "call", "variable", "let", "list", "mapping", "quantifier", "error", "missing"]));
assert(same([...E.DIALECT.kinds().values()].map((kind) => kind.role), [
  F.LITERAL, F.APPLICATION, F.REFERENCE, F.BINDING, F.APPLICATION, F.APPLICATION, F.QUANTIFIER, F.INVALID, F.INVALID]));
for (const domain of [D.Anything, new D.OfTypes("int", BigInt), new D.OfValues("odd", (v) => Number(v) % 2 === 1), new D.OfUnion(D.Anything)]) {
  assert(conformance_problems(domain.constructor as any, "Domain").length === 0);
}
for (const signature of [new D.Function([], D.Anything), new D.Same(1, [D.Anything]), new D.Overloaded(new D.Function([], D.Anything))]) {
  assert(conformance_problems(signature.constructor as any, "Signature").length === 0);
}
for (const translator of Translators.TRANSLATORS) {
  assert(translator instanceof FT.Pairwise && translator.inverse() instanceof FT.Pairwise
    && conformance_problems(FT.Pairwise, "Translator").length === 0);
}
// The package exports the framework alone; each dialect, Patterns among them, is imported from its own path.
const root: any = await import("@mbse/expressions");
assert(same(Object.keys(root), ["Framework"]) && root.Framework === Framework);

<a id="FRM-02"></a>
## FRM-02 · Forms and `make` are inverses, for every kind of every dialect

In [ ]:
for (const dialect of [...DIALECTS, ...C_FAMILY, ...POSIX]) {
  const kinds = new Set<string>();
  for (const node of F.walk(samples[dialect.name()])) {
    const form = F.form(node);
    kinds.add(form.kind);
    const made = term((READER.get(F.dialectOf(node)) as F.Dialect).make(form)); // made by the dialect whose union holds it
    assert(F.same(made, node) && F.form(made).kind === form.kind && made !== node);
  }
  if (dialect.reads().length > 0) { // SystemVerilog's sample holds its Types' too, its sublanguages' those of Expressions,
    // and the C family's those of the dialects each reads
    assert([...dialect.kinds().keys()].filter((tag) => !INVALID.has(tag)).every((tag) => kinds.has(tag)), dialect.name());
    continue;
  }
  assert(equal([...kinds].sort(), [...dialect.kinds().keys()].filter((tag) => !INVALID.has(tag)).sort()), dialect.name()); // every valid kind
}
// SystemVerilog's sample holds every kind of its Expressions and Types, and the Preprocessor's macro usage; a dialect of
// its family makes a kind of a dialect it reads, where its `Term` lists it, and refuses one it does not.
const kindsOf = (sample: any) => new Set([...F.walk(sample)].map((n) => F.kindOf(n)[0]));
assert(equal([...kindsOf(samples.SystemVerilog)].sort(), [...SV.TAGS, "integeratomtype", "macrousage"].sort()));
assert(SVT.TAGS.every((tag) => kindsOf(samples["SystemVerilog Types"]).has(tag)));
assert(form_equal(F.form(term(SVT.DIALECT.make(new F.Form("literal", { value: 8n })))), new F.Form("literal", { value: 8n }))); // Types' Term lists it
assert(SVT.DIALECT.kindNamed("literal") === SV.DIALECT.kinds().get("literal") && SV.DIALECT.kindNamed("structtype") === SVT.DIALECT.kinds().get("structtype"));
raises(ValueError, () => SVT.DIALECT.make(new F.Form("call")), "SystemVerilog Types has no kind 'call'");
assert(TST.TAGS.every((tag) => kindsOf(samples["TypeScript Types"]).has(tag))); // and TypeScript's alike
assert(TSP.TAGS.every((tag) => kindsOf(samples["TypeScript Patterns"]).has(tag)));
assert(form_equal(F.form(term(TST.DIALECT.make(new F.Form("literal", { value: 1.0 })), TST.DIALECT)), new F.Form("literal", { value: 1.0 })));
assert(TST.DIALECT.kindNamed("literal") === TS.DIALECT.kinds().get("literal") && TS.DIALECT.kindNamed("tsuniontype") === TST.DIALECT.kinds().get("tsuniontype"));
assert(TS.DIALECT.kindNamed("arraypattern") === TSP.DIALECT.kinds().get("arraypattern")); // TypeScript5's `Term` lists it
raises(ValueError, () => TSP.DIALECT.make(new F.Form("call")), "TypeScript Patterns has no kind 'call'");
assert(form_equal(F.form(term(3n)), new F.Form("literal", { value: 3n })));
assert(form_equal(F.form(term(E.let_("x", null, null))), new F.Form("let", { name: "x" }, [null, null]))); // empty subterms are null
assert(form_equal(F.form(term((u: any) => u.call((c: any) => c))), new F.Form("call")));
raises(ValueError, () => E.DIALECT.make(new F.Form("constant")), "Patterns has no kind 'constant'");
raises(ValueError, () => term(E.DIALECT.make(new F.Form("literal", { name: "x" }))), "a literal has no attribute 'name'");
raises(ValueError, () => term(E.DIALECT.make(new F.Form("let", { name: "x" }, [basic]))), "a let takes 2 arguments, got 1");
raises(ValueError, () => term(E.DIALECT.make(new F.Form("quantifier", { name: "x", quantifier: "all" }, [1n, 2n, 3n]))),
  "a quantifier takes 2 arguments, got 3");
const spelled = term((u: any) => u.literal((r: any) => r.text("08").problems(["leading zeros"]))); // a container of scalars too
assert(F.same(term(E.DIALECT.make(F.form(spelled))), spelled));
let made = term(E.DIALECT.make(new F.Form("call", { callable: "f" }, [E.variable("g"), 1n, 2n]))); // specs: writers, natives
assert(same(F.form(made).arguments.map((a) => F.kindOf(a)[0]), ["variable", "literal", "literal"]));
raises(ValueError, () => F.kindOf(term((u: any) => u)), "a term holds no dialect"); // nothing written: no branch of Term is selected
const alone = store.transaction(() => store.Expressions.Patterns.Expressions.Term((u: any) => u)); // a value of a dialect's `Term`, alone
assert(F.isTerm(alone) && F.dialectOf(alone) === E.DIALECT.union);
raises(ValueError, () => F.kindOf(alone), "a term holds no kind");
raises(ValueError, () => term(P.DIALECT.make(new F.Form(P.CALL)), P.DIALECT), "a call takes at least 1 arguments, got 0"); // its function, then its arguments
made = term(P.DIALECT.make(new F.Form(P.CALL, {}, [P.name("f"), P.constant(1n), 2n])), P.DIALECT); // a native: Python's constant
assert(F.nameOf(F.form(made).arguments[0]) === "f" && F.form(made).arguments.slice(1).every((a, i) =>
  form_equal(F.form(a), new F.Form(P.LITERAL, { value: BigInt(i + 1) }))));
assert(F.kindOf(term(1n, M.DIALECT))[0] === "literal" && F.kindOf(term(true, C.DIALECT))[0] === "literal"); // its literal kind
raises(TypeError, () => E.DIALECT.spec([1n]), "expected a term, a writer or a native value, got [1]"); // no literal kind holds these
raises(TypeError, () => X.DIALECT.spec(new Uint8Array([120])), "an Excel literal cannot hold a bytes");
// A form and a part print as Python writes them, in messages (both languages alike)
assert(Repr.repr(new F.Form("call", { callable: "add" }, ["x"])) === "Form(kind='call', attributes={'callable': 'add'}, arguments=('x',))");
assert(Repr.repr(new F.Part("keyword", { arg: "x" }, 1)) === "Part(kind='keyword', attributes={'arg': 'x'}, size=1)");

<a id="FRM-03"></a>
## FRM-03 · `walk`, `fold` and `same` traverse through forms alone

In [ ]:
const shared = E.variable("x");
const pair = term(E.operation("f", shared, shared));
assert(same([...F.walk(pair)].map((e) => F.form(e).kind), ["call", "variable", "variable"])); // a nested value: each visited
assert(equal([...F.walk(term(basic))].map((e) => F.form(e).attributes).slice(0, 3), map([{ name: "a" }, { callable: "get" }, { name: "person" }])));
assert(F.fold(pair, (node, results: number[]) => 1 + results.reduce((a, b) => a + b, 0)) === 3);
const calls: unknown[] = [];
F.fold(pair, (node) => calls.push(node));
assert(calls.length === 3); // once per term
assert(equal(F.fold(term(E.let_("x", null, 1n)), (node, results) => results), [null, []]));
for (const [a, b, expected] of [[1n, 1n, true], [1n, 1.0, false], [NaN, NaN, true], [-0, 0, false],
  [0, 0, true], ["x", "x", true], [new Uint8Array([120]), new Uint8Array([120]), true], [new Uint8Array([120]), new Uint8Array([121]), false]] as const) {
  assert(F.same(term(a), term(b)) === expected, `${String(a)} ${String(b)}`);
}
assert(F.same(term(basic), term(E.DIALECT.make(F.form(term(basic))))) && F.same(term(E.let_("x", null, null)), term(E.let_("x", null, null))));
assert(!F.same(term(E.let_("x", null, null)), term(E.let_("x", 1n, 2n))) && !F.same(term(1n), term(E.variable("x"))));
assert(!F.same(term(E.operation("f", 1n)), term(E.operation("f", 1n, 2n))));
assert(!F.same(term(E.operation("f")), term((u: any) => u.call((c: any) => c)))); // attributes differ

<a id="FRM-04"></a>
## FRM-04 · Every dialect's install adds its union of its kinds to `Term`, and its terms round-trip

In [ ]:
for (const dialect of DIALECTS) {
  assert(Term.branches.some((b: any) => b.name === dialect.tag));
  const sample = samples[dialect.name()];
  for (const [encode, decode] of [[JSON.ToJSON(store), JSON.FromJSON(store)], [YAML.ToYAML(store), YAML.FromYAML(store)]] as [any, any][]) {
    const back = decode(Term, encode(Term, sample));
    assert(F.same(back, sample) && dialect.validate(term(back, dialect), ["person"]).length === 0); // held, as validation reads its store
  }
}
// The framework reads a kind's subterms, variadic arguments and attributes from its schema: a subterm's schema is Term, or a
// dialect's union.
const of = new Map<string, Map<string, any>>(DIALECTS.map((d) => [d.name(),
  new Map<string, any>(store.registered(d.union).branches.map((b: any) => [b.name, b.schema]))]));
const kindSchemas = of.get("Patterns") as Map<string, any>;
assert(equal(F._parts(kindSchemas.get("let")), [["value", "body"], [], ["name"]]));
assert(equal(F._parts(kindSchemas.get("call")), [[], ["arguments"], ["callable"]]));
assert(equal(F._parts(kindSchemas.get("quantifier")), [["value", "body"], [], ["name", "quantifier"]])); // a binder: Let's first
assert(equal(F._parts(kindSchemas.get("literal")), [[], [], ["value", "text", "problems"]]));
assert(equal(F._parts(kindSchemas.get("list")), [[], ["elements"], []]) && equal(F._parts(kindSchemas.get("mapping")), [[], ["entries"], []]));
assert(equal(F._parts(kindSchemas.get("error")), [[], ["terms"], ["text"]]) && equal(F._parts(kindSchemas.get("missing")), [[], [], ["kind"]]));
assert(equal(F._members(kindSchemas.get("mapping"), "entries"), ["key", "value"])
  && F._members(kindSchemas.get("list"), "elements").length === 0);
for (const dialect of DIALECTS.slice(1)) {
  for (const tag of dialect.kinds().keys()) { // a dialect's own kinds are its schemas, named by it
    const name: string = of.get(dialect.name())?.get(tag).name;
    assert(name.startsWith(`Expressions.${dialect.path}.`) || name.startsWith("Expressions.Framework.")
      || (dialect === TST.DIALECT && tag === "identifier"), tag); // but TypeScript Types' identifier, TypeScript5's, read so
  }
}
const python3 = of.get("Python3") as Map<string, any>;
assert(same([...P.DIALECT.kinds().keys()].map((tag) => python3.get(tag).name).slice(-10), [
  "Expressions.Python.Python3.Compare", "Expressions.Python.Python3.BoolOp", "Expressions.Python.Python3.NamedExpr", "Expressions.Python.Python3.IfExp",
  "Expressions.Python.Python3.Lambda", "Expressions.Framework.Let", "Expressions.Python.Python3.Import", "Expressions.Python.Python3.ImportFrom",
  "Expressions.Framework.Error", "Expressions.Framework.Missing"]));
assert(python3.get(P.LITERAL).refines === store.registered("Expressions.Framework.Literal")); // the framework's, refined
assert(equal(F._parts(python3.get(P.CALL)), [["func"], ["args"], ["keywords"]])); // a subterm, variadic arguments, then helpers
assert(equal(F._helpers(python3.get(P.CALL)), ["keywords"]) && equal(F._helpers(python3.get(P.BINOP)), []));
assert(equal(F._parts(python3.get(P.GENERATOREXP)), [["value", "body"], [], ["name", "generators"]])); // a binder: Let's value empty
assert(equal(F._parts(python3.get(P.DICT)), [[], ["items"], []]) && equal(F._members(python3.get(P.DICT), "items"), ["key", "value"]));
assert(equal(F._parts(of.get("MATLAB")?.get("binary")), [["left", "right"], [], ["operator"]]));
assert(equal(F._parts(of.get("Excel")?.get("cell")), [[], [], ["address", "sheet", "book"]]));
assert(python3.get(P.CALL).properties.get("func").schema === store.registered(P.DIALECT.union)); // its own union
assert(of.get("C")?.get("binaryexpression").properties.get("left").schema === store.registered(C.DIALECT.union)); // C's kinds hold C's terms
assert(of.get("Cpp")?.get("binaryexpression").properties.get("left").schema === store.registered(CPP.DIALECT.union)); // and C++'s, C++'s
// A kind that binds names refines the framework's `Let` (a binder): SystemVerilog's array method with a `with` clause, its
// call Let's value and its `with` expression Let's body, each redeclared to hold its dialect's terms.
const [LET, WITH] = [store.registered("Expressions.Framework.Let"), of.get("SystemVerilog")?.get("arraymethodwithexpression")];
assert(WITH.refines === LET && F.isBinder(WITH) && F.isBinder(LET) && !F.isBinder(of.get("SystemVerilog")?.get("callexpression")));
assert(!F.isBinder(store.registered("Schemas.OfInt")) && equal(F._parts(WITH), [["value", "body"], [], ["name"]]));
assert(WITH.properties.get("value").schema === store.registered(SV.DIALECT.union) && WITH.properties.get("body").schema === store.registered(SV.DIALECT.union));
// Each dialect's holder, the root of an expression tree: a reference object refining the framework's Expression, its
// root narrowed to the dialect's Term. It validates, holds a term of its dialect, round-trips, prints (FRM-11), and HasRoot,
// which links the framework's Expression, links it.
const FrameworkExpression: any = store.registered("Expressions.Framework.Expression");
const holders = new Map<F.Dialect, any>();
for (const dialect of DIALECTS) {
  const Holder: any = store.registered(`Expressions.${dialect.path}.Expression`);
  assert(Holder.ref && Holder.refines === FrameworkExpression && Holder.validate().length === 0 && F.isExpression(Holder));
  assert(Holder.properties.get("root").schema === store.registered(dialect.union)); // narrowed from Expression's Term
  const sample = samples[dialect.name()];
  const [held, linked] = store.transaction(() => {
    const held = dialect.path.split(".").reduce((at: any, part) => at[part], store.Expressions).Expression((e: any) => // C's and C++'s name their ABI
      (["c", "cpp"].includes(dialect.tag) ? e.root(dialect.spec(sample)).abi([AMD64_]) : e.root(dialect.spec(sample))));
    const linked = store.Schemas.OfExpression((s: any) => s.name(`frm.${dialect.name()}`).expression(
      (a: any) => a.insert((e: any) => e.expression(held))));
    return [held, linked];
  });
  holders.set(dialect, held);
  assert(F.same(F.term(held), sample) && F.dialectOf(F.term(held)) === dialect.union);
  assert(Validators.Validate(store)(Holder, held).length === 0 && dialect.validate(held, ["person"]).length === 0);
  assert(F.same((JSON.FromJSON(store)(Holder, JSON.ToJSON(store)(Holder, held)) as any).root, held.root));
  assert(same([...(new Builders.OfExpression.Builder(linked, false, store) as any).expression].map((entry: any) => F.same(entry.expression, held)), [true]));
}
// Expression names the ABI its terms are read under, one at most, through its adjacency abi (HasAbi), which every
// dialect's holder inherits: C's and C++'s read the types their standards leave to the implementation from it
const HasAbi: any = store.registered("Expressions.Framework.HasAbi");
assert(same([...FrameworkExpression.adjacencies.keys()], ["abi"]) && FrameworkExpression.adjacencies.get("abi").relation === HasAbi
  && FrameworkExpression.adjacencies.get("abi").me === "expression");
assert(same([...HasAbi.entry.links.keys()], ["expression", "abi"]) && HasAbi.entry.links.get("abi").schema === Dialects);
assert(same([...store.registered("Expressions.C.Expressions.Expression").adjacencies.keys()], ["abi"]));
assert(same([...holders.get(C.DIALECT).abi].map((entry: any) => entry.abi), [AMD64_]) && [...holders.get(E.DIALECT).abi].length === 0);
raises(Error, () => store.transaction(() => store.Expressions.Python.Python3.Expression((e: any) => e.root(E.list_(1n).all("p", true)))),
  "'quantifier' is not a member of a union value"); // its root holds its dialect's terms alone

<a id="FRM-05"></a>
## FRM-05 · Validation follows each kind's role, in every dialect

In [ ]:
assert(DIALECTS.every((d) => d.validate(samples[d.name()], ["person"], true).length === 0));
assert(same(E.DIALECT.validate(term(E.operation("add", 1n))), ["add takes 2 arguments, got 1"]));
assert(same(E.DIALECT.validate(term(E.operation("not", E.variable("x")))), ["argument 0: variable 'x' is not bound"]));
assert(same(E.DIALECT.validate(term(E.mapping([E.variable("k"), 1n], ["b", E.list_(E.variable("x"))]))), [
  "key 0: variable 'k' is not bound", "value 1: argument 0: variable 'x' is not bound"])); // a mapping's terms, by entry
assert(same(E.DIALECT.validate(term((u: any) => u.mapping((m: any) => m.entries([(a: any) => a.as_object((o: any) => o.key(E.literal(1n)))])))), [
  "a mapping needs a value 0"]));
assert(same(E.DIALECT.validate(term(E.let_("a", E.variable("a"), E.variable("a")))), ["value: variable 'a' is not bound"]));
assert(same(E.DIALECT.validate(term((u: any) => u.let((r: any) => r.name("x").value(E.literal(1n))))), ["a let needs a body"]));
for (const [spec, bound, problems] of [
  [P.subscript(P.name("t"), "x"), [], ["value: variable 't' is not bound"]],
  [P.call("f", 1n), [], ["func: variable 'f' is not bound"]], // a subterm, then variadic arguments
  [P.call(P.name("f"), P.name("x")), ["f"], ["argument 0: variable 'x' is not bound"]],
  [P.call(P.name("f"), P.keyword("k", P.name("x"))), ["f"], ["keywords 0 value: variable 'x' is not bound"]], // a helper's
  [(u: any) => u[P.CALL]((r: any) => r.func(P.name("f")).keywords([(a: any) => a.as_object((o: any) => o.arg("k"))])),
    ["f"], ["a call needs a keywords 0 value"]],
  [P.listcomp(P.name("y"), P.comprehension("y", P.name("y"))), [], ["generators 0 iter: variable 'y' is not bound"]],
  [P.lambda_(P.name("a"), ["a", P.arg("b", P.name("a"))]), [], ["args args 1 default_value: variable 'a' is not bound"]],
  [P.subscript(P.name("t"), P.slice_()), ["t"], []], // a kind's optional subterms may be empty
  [P.import_("numpy", P.name("np"), "np"), [], []], // an import binds a name; its alias is optional
  [P.import_("numpy", P.name("np")), [], ["body: variable 'np' is not bound"]],
  [(u: any) => u[P.IMPORT]((r: any) => r.module("numpy")), [], ["an import needs a body"]],
  [P.name("not a name"), [], ["a name must be an identifier, got 'not a name'"]], // a kind's own check
] as [unknown, string[], string[]][]) {
  assert(same(P.DIALECT.validate(term(spec, P.DIALECT), bound), problems));
}
assert(X.DIALECT.validate(term(X.cell("A1"), X.DIALECT)).length === 0
  && X.DIALECT.validate(term(X.cell("B2", "Sheet1"), X.DIALECT), [], true).length === 0); // cells need no binding
assert(same(M.DIALECT.validate(term((u: any) => u.field((r: any) => r), M.DIALECT)), ["a field needs a name", "a field needs a value"]));
assert(same(M.DIALECT.validate(term((u: any) => u.binary((r: any) => r.operator("&&").left(M.constant(1n))), M.DIALECT)), ["a binary needs a right"]));
assert(M.DIALECT.validate(term(M.binary("**", 1n, 2n), M.DIALECT)).length === 0
  && same(M.DIALECT.validate(term(M.binary("**", 1n, 2n), M.DIALECT), [], true), ["'**' is not a core operation"]));
assert(M.DIALECT.validate(term(M.field(M.identifier("s"), "any name"), M.DIALECT), ["s"], true).length === 0); // a field's vocabulary is open
assert(same(X.DIALECT.validate(term(X.function("NOT", true, false), X.DIALECT)), ["NOT takes 1 arguments, got 2"]));
assert(X.DIALECT.validate(term(X.function("AND", true), X.DIALECT)).length === 0); // AND: any number
assert(same(X.DIALECT.validate(term(X.let_("a", X.name("a"), X.name("a")), X.DIALECT)), ["value: variable 'a' is not bound"]));
assert(same(X.DIALECT.validate(term((u: any) => u.literal((r: any) => r.value((v: any) => v.as_scalar(new Uint8Array([120])))), X.DIALECT)),
  ["an Excel literal cannot hold a bytes"])); // the dialect's own check of the framework's literal
assert(same(M.DIALECT.validate(term((u: any) => u.call((r: any) => r.callable(7n)), M.DIALECT)), ["a call's callable must be a str, got int"]));

<a id="FRM-06"></a>
## FRM-06 · Domains and signatures

In [ ]:
{
  const [Int, Float, Str] = [new D.OfTypes("int", BigInt), new D.OfTypes("float", Number), new D.OfTypes("str", String)];
  const Odd = new D.OfValues("odd", (v) => Number(v) % 2 === 1);
  const Number_ = new D.OfUnion(Int, Float);
  assert(Int.contains(1n) && !Int.contains(true) && !Int.contains(1.0) && String(Int) === "int");
  assert(new D.OfTypes("bool", Boolean).contains(true) && !new D.OfTypes("bool", Boolean).contains(1n));
  assert(Str.contains("a") && !Str.contains(new Uint8Array([97])));
  assert(Number_.name() === "int | float" && String(Number_) === "int | float" && Number_.contains(1.5) && !Number_.contains("1"));
  assert(Number_.includes(Int) && Number_.includes(new D.OfUnion(Float, Int)) && !Int.includes(Number_));
  assert(new D.OfTypes("num", BigInt, Number).includes(Number_) && !Number_.includes(Str) && !Number_.includes(D.Anything));
  assert(Odd.contains(3n) && Odd.includes(Odd) && !Odd.includes(Int) && Odd.includes(new D.OfUnion(Odd)) && String(Odd) === "odd");
  assert(D.Anything.includes(Number_) && D.Anything.contains(null) && String(D.Anything) === "any" && D.Anything.name() === "any");
  assert(D.overlaps(Int, D.Anything) && D.overlaps(Number_, Int) && !D.overlaps(Int, Str));
  assert(new D.OfTypes("list", Array).contains([1n]) && !new D.OfTypes("list", Array).contains("x")); // a class, by instance
  const f = new D.Function([Int, Str], Float);
  assert(f.arity() === 2 && f.result([Int, Str]) === Float && f.result([D.Anything, Str]) === Float);
  assert(f.result([Str, Str]) === null && f.result([Int]) === null && f.describe() === "(int, str) -> float");
  assert(f.exact([Int, Str]) && !f.exact([D.Anything, Str]));
  const sameOf = new D.Same(2, [Int, Float]);
  assert(sameOf.result([Int, Int]) === Int && sameOf.result([Int, Float]) === null && sameOf.result([Int]) === null);
  assert(sameOf.result([D.Anything, D.Anything])?.name() === "int | float" && sameOf.result([D.Anything, Float]) === Float);
  assert(sameOf.describe() === "(T, T) -> T for T in int | float" && new D.Same(1, [Int], Str).describe() === "(T) -> str for T in int");
  assert(new D.Same(2, [Int, Float], Str).result([D.Anything, D.Anything]) === Str); // one result, however many match
  assert(new D.Opaque().arity() === -1 && new D.Opaque(Int).result([Str, Str, Str]) === Int && new D.Opaque().describe() === "(...) -> any");
  const either = new D.Either(2);
  assert(either.arity() === 2 && either.result([Int, Int]) === Int && either.result([Int, Str])?.name() === "int | str");
  assert(either.result([Int]) === null && either.describe() === "(T, T) -> T");
  const anyNumber = new D.Either(-1); // Python's `and` of any number of values
  assert(anyNumber.result([Int, Str, Int])?.name() === "int | str" && anyNumber.result([]) === null && anyNumber.describe() === "(T, ...) -> T");
  assert(sameOf.exact([Int, Int]) && !sameOf.exact([Int, Float]) && !sameOf.exact([Int]));
  const over = new D.Overloaded(new D.Same(2, [Int]), new D.Function([Number_, Number_], Float));
  assert(over.arity() === 2 && over.result([Int, Int]) === Int && over.result([Int, Float]) === Float);
  assert(over.result([D.Anything, Int])?.name() === "int | float" && over.result([Str, Str]) === null);
  assert(over.describe() === "(T, T) -> T for T in int | (int | float, int | float) -> float");
}

<a id="FRM-07"></a>
## FRM-07 · `infer` finds an expression's domain from its vocabulary's signatures

In [ ]:
function infer(spec: unknown, environment: Record<string, any> = {}) {
  return E.DIALECT.infer(term(spec), environment);
}

assert(infer(basic, { person: BD.Object }) === BD.Bool);
assert(infer(E.literal(1n).add(2n)) === BD.Int && infer(E.literal(1.5).neg()) === BD.Float);
assert(infer(person.age, { person: BD.Object }) === D.Anything);
assert(infer(person.age.add(1n), { person: BD.Object }) === BD.Int); // only int + int applies
assert(M.DIALECT.infer(term(M.call("dist", 1n, 2n), M.DIALECT)) === D.Anything); // an extension: nothing is known of it
raises(ValueError, () => infer(E.operation("contains", 1n, 2n)),
  "cannot infer the domain of an invalid expression: 'contains' is not an operation Patterns defines"); // Patterns' operations are the ones it defines
assert(infer(E.list_(1n, 2n)).name() === "list(int)" && infer(E.list_(1n, "a")).name() === "list(int | str)");
assert(infer(E.list_()).name() === "list(any)" && infer(E.list_(1n, 2n).count()) === BD.Int);
assert(infer(E.mapping(["a", 1.5])).name() === "keyed(str, float)" && infer(E.mapping()).name() === "keyed(any, any)");
assert(infer(E.list_(1n, 2n).item(0n)) === BD.Int && infer(E.list_(1n, 2n).all("x", E.variable("x").gt(0n))) === BD.Bool);
assert(BD.LIST.describe() === "(T, ...) -> list(T)" && BD.MAPPING.describe() === "(K, T, ...) -> keyed(K, T)" && BD.LIST.arity() === -1);
assert(infer(E.let_("a", 1n, E.let_("b", "s", E.variable("a")))) === BD.Int);
raises(TypeError, () => infer(E.literal(1n).add(1.5)), "add cannot take (int, float); it takes (T, T) -> T for T in int | float");
raises(TypeError, () => infer(E.literal(1n).has("x")), "has cannot take (int, str); it takes (object | record, str) -> bool");
raises(ValueError, () => infer(person), "cannot infer the domain of an invalid expression: variable 'person' is not bound");
assert(BD.of(term(1n)) === BD.Object && BD.of(new Uint8Array()) === BD.Bytes); // a term value is an object
raises(TypeError, () => BD.of([]), "a list is not a value of the Patterns dialect");
assert(M.DIALECT.infer(samples.MATLAB, { person: D.Anything }).name() === "logical");
assert(X.DIALECT.infer(samples.Excel, { person: D.Anything }).name() === "logical");
assert(L.DIALECT.infer(samples.LaTeX, { person: D.Anything }).name() === "truth");
assert(P.DIALECT.infer(samples.Python3, { person: D.Anything }) === D.Anything); // a call gives what is not known
assert(TS.DIALECT.infer(samples.TypeScript5, { person: D.Anything }) === D.Anything); // nothing is known but a literal's
assert(P.DIALECT.infer(term(P.boolop("and", 1n, "a"), P.DIALECT)).name() === "int | str"); // Python's and gives an operand
assert(P.DIALECT.infer(term(P.import_("math", P.binop("+", 1n, true)), P.DIALECT)).name() === "int");

<a id="FRM-08"></a>
## FRM-08 · The interpreter evaluates by role and raises what validation reports

In [ ]:
/** A simplifier by `rules`, its literals a dialect's natives'. */
const simplifier = (rules: FV.Rules, dialect: F.Dialect = E.DIALECT) => new FV.Simplifier(rules, { literal: FV.natives(dialect) });
const simplify = simplifier(new FV.Interpreter(E.DIALECT, new Map<string, any>([["call", new Map()], ["quantifier", new Map()]])));
raises(TypeError, () => simplify.run(3n), "not a term: 3");
raises(ValueError, () => simplify.run(term(E.literal(null))), "a literal needs a value");
const unbound = simplify.run(term(E.variable("x"))); // not bound: it stays
assert(same([...unbound.free], ["x"]) && !unbound.literal && unbound.residues.length === 0);
const kept = simplify.run(term(E.literal(true).and_(false))); // no rule for and: it stays, named
assert(equal(kept.residues.map((r) => [r.dialect, r.kind, r.what, r.category]), [["Patterns", "call", "call 'and'", "a"]]));
assert((kept.residues[0] as FV.Residue).reason === "Patterns call 'and' is not simplified" && !kept.literal);
const typed = simplifier(new FV.Interpreter(C.DIALECT, new Map()), C.DIALECT).run(term(C.literal(5n, "u"), C.DIALECT)); // a kind's own domain: C's type
assert(same(typed.residues.map((r) => r.reason), ["C literal of unsigned int is not simplified"]));
raises(TypeError, () => simplifier(new FV.Interpreter(E.DIALECT, new Map([["call", new Map([["not", null as any]])]]))).run(term(E.operation("not", 1n, 2n))),
  "not takes 1 arguments, got 2");
raises(ValueError, () => V.OfAny(term(E.let_("a", 1n, null))), "a let needs a body");
raises(ValueError, () => V.OfAny(term(E.quantifier("all", "x", null, true))), "a quantifier needs a value");
assert(simplifier(new FV.Interpreter(E.DIALECT, new Map([["call", new Map()]]), { extension: (operator) => BigInt(operator.length) }))
  .run(term(E.operation("dist", 1n, 2n))).value === 4n); // operators outside the vocabulary go to the extension

const scope = new SY.Variables({ x: 1n });
let x: any = term(E.variable("x"));
assert(conformance_problems(SY.Variables, "Scope").length === 0 && scope.lookup(x) === 1n);
const inner = scope.bind("x", 2n);
assert(inner.lookup(x) === 2n && scope.lookup(x) === 1n); // bind makes a new scope
raises(SY.Unbound, () => scope.lookup(term(E.variable("y"))), "y"); // which a simplifier catches, leaving the reference
raises(SY.Unbound, () => scope.bind("x", SY.UNKNOWN).lookup(x), "x"); // a name hidden
assert(V.OfAny(x, inner).value === 2n); // a simplifier takes a scope as well as an object of variables

<a id="FRM-09"></a>
## FRM-09 · Symbolics: the names an expression needs, and the imports it declares

In [ ]:
const sorted = (names: Set<string>) => [...names].sort();
assert(same(sorted(SY.free(term(basic), E.DIALECT)), ["person"]) && SY.free(term(1n), E.DIALECT).size === 0);
assert(same(sorted(SY.free(term(E.let_("a", E.variable("a"), E.variable("a"))), E.DIALECT)), ["a"])); // the name is bound in the body only
assert(same(sorted(SY.free(term(E.let_("a", null, E.variable("b"))), E.DIALECT)), ["b"])); // empty subterms are skipped
assert(same(sorted(SY.free(term(E.quantifier("all", "x", E.variable("xs"), E.variable("x").gt(E.variable("y")))), E.DIALECT)), ["xs", "y"]));
assert(SY.imports(term(basic), E.DIALECT).length === 0);
for (const [error, name] of [[FE.IndexError, "IndexError"], [FE.OverflowError, "OverflowError"],
  [FE.ZeroDivisionError, "ZeroDivisionError"]] as const) {
  assert(new error("m") instanceof Error && new error("m").name === name); // Python's exceptions, by the same names
}
assert(!("NameError" in FE) && !("ImportError" in FE)); // a name or an import a scope lacks stays: no error
assert(same(sorted(SY.free(samples.Python3, P.DIALECT)), ["person"])); // np, f and a are bound by imports and the let
assert(same(sorted(SY.free(samples.TypeScript5, TS.DIALECT)), ["person"])); // p by its arrow; properties and keys name, not refer
assert(same(sorted(SY.free(term(P.call("hasattr", P.name("x"), "k"), P.DIALECT), P.DIALECT)), ["x"])); // builtins are ambient
assert(same(sorted(SY.free(term(X.infix("+", X.cell("A1"), X.name("r")), X.DIALECT), X.DIALECT)), ["r"])); // cells are resolved by the workbook, not bound
assert(same(sorted(SY.free(term(M.import_("pkg.*", M.identifier("x")), M.DIALECT), M.DIALECT)), ["x"])); // MATLAB's imports bind functions, not variables
assert(same(SY.imports(samples.Python3, P.DIALECT).map((i) => F.kindOf(i)[0]), ["import", "importfrom"]));
assert(same(SY.imports(samples.MATLAB, M.DIALECT).map((i) => F.nameOf(i)), ["pkg.*"]));

<a id="FRM-10"></a>
## FRM-10 · A dialect may extend another: its kinds are the base's and its own, added to its union by schema extension

In [ ]:
/** An extension of Patterns, added to its union: `twice(x)`, x + x; `import(module, body)`, an import binding the
 * module's name within its body; `cell`, a reference the scope resolves without a binding; and `apply(function,
 * arguments)`, a subterm and variadic arguments. And `Bare`, a dialect of its own: its `Term`, `Expressions.Bare.Term`,
 * the framework's literal and the import, the branch `bare` of Term. */
function install(txn: any): any {
  const S = txn.Schemas;
  const Framework = txn.Expressions.Framework;
  return txn.uid("frm.Ext/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Term: S.OfUnion, Patterns: S.OfUnion, Bare: S.OfUnion, Twice: S.OfObject, Import: S.OfObject, Cell: S.OfObject,
      Apply: S.OfObject })
    .spec([
      (_: any) => _.Twice.name("frm.Twice").properties({ operand: _.Term, domain: S.OfStr }),
      (_: any) => _.Import.name("frm.Import").properties({ name: S.OfStr, body: _.Term, domain: S.OfStr }),
      (_: any) => _.Cell.name("frm.Cell").properties({ name: S.OfStr, domain: S.OfStr }),
      (_: any) => _.Apply.name("frm.Apply").properties({
        function: _.Term, arguments: S.OfContainer((c: any) => c.index([S.OfInt]).element([_.Term])), domain: S.OfStr }),
      (_: any) => _.Patterns.name("Expressions.Patterns.Expressions.Term").branches([ // Patterns' union, extended
        (t: any) => t.name("twice").schema([_.Twice]), (t: any) => t.name("import").schema([_.Import]),
        (t: any) => t.name("cell").schema([_.Cell]), (t: any) => t.name("apply").schema([_.Apply])]),
      (_: any) => _.Bare.name("Expressions.Bare.Term").branches([
        (t: any) => t.name("literal").schema([Framework.Literal]), (t: any) => t.name("import").schema([_.Import])]),
      (_: any) => _.Term.name("Expressions.Framework.Term").branches([(t: any) => t.name("bare").schema([_.Bare])]),
    ])]))]);
}

store.transaction(install);
const Ext = new F.Dialect("Ext", {
  twice: new F.Kind(F.APPLICATION, { signature: new D.Function([BD.Int], BD.Int) }), // an open vocabulary: its tag names it
  import: new F.Kind(F.IMPORT, { required: ["name"], names: (attributes) => [attributes.get("name")] }),
  cell: new F.Kind(F.REFERENCE, { required: ["name"], lexical: false }),
  apply: new F.Kind(F.APPLICATION, { signature: new D.Opaque() }),
}, { domain_of: BD.of, extends: E.DIALECT });
assert(same([...Ext.kinds().keys()], [...E.DIALECT.kinds().keys(), "twice", "import", "cell", "apply"]) && Ext.base === E.DIALECT && E.DIALECT.base === null);
assert(Ext.tag === E.DIALECT.tag && Ext.union === E.DIALECT.union); // its terms are its base's union's
raises(ValueError, () => term(Ext.make(new F.Form("apply")), Ext), "an apply takes at least 1 arguments, got 0");
const twice = (operand: unknown) => new E.Writer((u: any) => u.twice((r: any) => r.operand(Ext.spec(operand))));
const import_ = (name: string, body: unknown, dialect: F.Dialect = Ext) => new F.Writer(
  (u: any) => u.import((r: any) => r.name(name).body(dialect.spec(body))), dialect.tag);
const cell = (name: string) => new F.Writer((u: any) => u.cell((r: any) => r.name(name)), Ext.tag);
x = E.variable("x");
const mixed = term(x.add(twice(x)), Ext); // a writer of the base holds the extension's term
assert(Ext.validate(mixed, ["x"]).length === 0 && F.kindOf(F.form(mixed).arguments[1])[0] === "twice");
assert(same(E.DIALECT.validate(mixed, ["x"]), ["argument 1: Patterns has no kind 'twice'"])); // the base reads only its own
raises(ValueError, () => V.OfAny(mixed, { x: 1n }).value, "Patterns has no kind 'twice'");

/** A scope with modules to import and cells to read. */
class Workbook extends SY.Variables {
  readonly cells: Map<string, unknown>;
  readonly modules: Map<string, unknown>;

  constructor(variables: SY.Bindings = {}, cells: Record<string, unknown> = {}, modules: Record<string, unknown> = {}) {
    super(variables);
    this.cells = new Map(Object.entries(cells));
    this.modules = new Map(Object.entries(modules));
  }

  override lookup(reference: any): any {
    if (F.kindOf(reference)[0] === "cell") return this.cells.get(F.nameOf(reference));
    return super.lookup(reference);
  }

  override enter(declaration: any): SY.Scope {
    const name = F.nameOf(declaration) as string;
    if (!this.modules.has(name)) throw new SY.Unbound(name);
    return this.bind(name, this.modules.get(name));
  }
}

const interpreter = new FV.Interpreter(Ext, new Map<string, any>([["call", V.OPERATIONS], ["quantifier", V.QUANTIFIERS],
  ["twice", (thunks: FV.Thunk[]) => thunks[0]!() * 2n]]), { scope: (variables) => new Workbook(variables) });
const extended = new FV.Simplifier(interpreter, { literal: (value) => (typeof value === "bigint" ? E.literal(value) : null) });
assert(extended.run(mixed, { x: 3n }).value === 9n);
const program = term(import_("m", E.variable("m").get("k").add(cell("A1"))), Ext);
const unknown = extended.run(program, new Workbook({}, { A1: 1n }, { m: term(E.literal(1n)) })); // no property k: unknown
assert(unknown.value === null && !unknown.literal);
assert(Ext.validate(program).length === 0 && SY.free(program, Ext).size === 0); // the import binds m; a cell needs no binding
assert(same(SY.imports(program, Ext).map((i) => F.nameOf(i)), ["m"]));
assert(Ext.infer(program).name() === "int | float" && Ext.infer(mixed, { x: BD.Int }) === BD.Int);
raises(SY.Unbound, () => new SY.Variables().enter(program), "import"); // Variables has no imports: an import stays
raises(ValueError, () => extended.run(term((u: any) => u.import((r: any) => r.name("m")), Ext)), "an import needs a body");
assert(same(Ext.validate(term((u: any) => u.import((r: any) => r.body(E.variable("m"))), Ext)), ["an import needs a name", "body: variable 'm' is not bound"]));
// Simplifying an extension's terms: an import the scope does not provide stays, binding its names within its body.
const needsY = term(import_("m", E.variable("m").get("k").add(E.variable("y"))), Ext);
assert(F.same(term(extended.run(needsY, { x: 1n }).residual, Ext), needsY));
assert(F.same(term(extended.run(term(twice(x).add(twice(E.variable("y"))), Ext), { x: 2n }).residual, Ext), term(E.literal(4n).add(twice(E.variable("y"))), Ext)));
const provided = extended.run(needsY, new Workbook({ y: 2n }, {}, { m: term(E.literal(1n)) })); // provided: entered, its body simplified
assert(provided.value === null && F.same(term(provided.residual, Ext), term(import_("m", E.variable("m").get("k").add(2n)), Ext)));
// A dialect's kinds need no checks, domains or names of their own.
const Bare = new F.Dialect("Bare", { literal: new F.Kind(F.LITERAL), import: new F.Kind(F.IMPORT) }, { domain_of: BD.of });
const three = (u: any) => u.literal((r: any) => r.value((p: any) => p.as_scalar(3n))); // Bare has no writers
const bare = term(import_("m", three, Bare), Bare);
assert(Bare.validate(bare).length === 0 && SY.free(bare, Bare).size === 0 && Bare.infer(bare) === BD.Int);
assert(new FV.Simplifier(new FV.Interpreter(Bare, new Map()), { literal: () => null }).run(term(three, Bare)).value === 3n); // kept as written
raises(ValueError, () => new F.Dialect("Clash", { literal: new F.Kind(F.APPLICATION) }, { domain_of: BD.of, extends: E.DIALECT }),
  "Clash would declare 'literal', which Patterns has");
raises(ValueError, () => new F.Dialect("Clash", { error: new F.Kind(F.APPLICATION) }, { domain_of: BD.of }),
  "Clash would declare 'error', which the framework has"); // every dialect reads invalid syntax

<a id="FRM-11"></a>
## FRM-11 · Invalid syntax, `error` and `missing`: every dialect's `Term`, `Type` and `Pattern` hold them, validation reports them, evaluation refuses them by name, and printers write them as written

In [ ]:
const PRINTERS = new Map<F.Dialect, (expression: unknown) => string>([[P.DIALECT, Python3.Text.ToText],
  [M.DIALECT, Matlab.Text.ToText], [X.DIALECT, Excel.Text.ToText], [L.DIALECT, Latex.Text.ToText],
  [C.DIALECT, CDialect.Text.ToText], [CPP.DIALECT, CppDialect.Text.ToText], [SV.DIALECT, SystemVerilog.Text.ToText],
  ...[SVTypes, SVPatterns, SVAssertions, SVConstraints, SVPreprocessor].map((pkg: any): [F.Dialect, (e: unknown) => string] =>
    [pkg.Expressions.DIALECT, pkg.Text.ToText]), // its family's, one printer
  [TS.DIALECT, TypeScript5.Text.ToText], [TST.DIALECT, TypeScriptTypes.Text.ToText],
  [TSP.DIALECT, TypeScriptPatterns.Text.ToText], [PYT.DIALECT, PythonTypes.Text.ToText], [PYP.DIALECT, PythonPatterns.Text.ToText],
  [PYR.DIALECT, PythonRegExp.Text.ToText], [TSR.DIALECT, TypeScriptRegExp.Text.ToText]]); // Patterns has none
assert([SVTypes, SVPatterns, SVAssertions, SVConstraints, SVPreprocessor].every((pkg: any) => pkg.Text.ToText === SystemVerilog.Text.ToText));
assert(TypeScriptTypes.Text.ToText === TypeScript5.Text.ToText && TypeScriptPatterns.Text.ToText === TypeScript5.Text.ToText); // its
// family's, one printer
for (const [dialect, write] of PRINTERS) assert(write(holders.get(dialect)) === write(samples[dialect.name()])); // FRM-04's
assert(TypeScript5.Text.ToText(samples.TypeScript5) === ( // every kind, a body of statements written `{}`
  "({ a: [person,, ...person], d: { person }, person = 1 } = person), async function* g<T>(x: T = this.y) {}, as" +
  "ync <U,>(p: U, ...r: U[]) => await import.meta.resolve(p), @person class K<T> extends person.Base<T> implements person {}, new" +
  " person.C<number>(...person), person`a${person}b`, `x${person}`, person++, typeof person, #p in person, person" +
  " ?? null, person ? /x/g : 1n, yield* person, person as string, person satisfies object, <any>person, person!, " +
  "person<string>, [1,, ...person], { m(): void, ...person, [person]: \"s\" }, <div id=\"x\" {...person}>{}{person}<>" +
  "text</></div>, <><br /></>, person?.a, super.x, import(person)"));
/** A name used as an expression, as each dialect's grammar makes it: TypeScript5's identifier, C++'s idexpression
 * holding its identifier, the framework's variable. */
function recovered(dialect: F.Dialect): (v: any) => any {
  if (dialect === PYR.DIALECT || dialect === TSR.DIALECT) { // a character, each RegExp dialect's own: its Term lists no
    return (v: any) => v.character((w: any) => w.value(97n)); // kind of its family's
  }
  if (TS_FAMILY.includes(dialect)) return (v: any) => v.identifier((w: any) => w.name("a")); // in each of TypeScript's unions
  if (dialect === CPP.DIALECT) return (v: any) => v.idexpression((w: any) => w.name((n: any) => n.identifier((i: any) => i.spelling("a"))));
  if (dialect === PYP.DIALECT) return (v: any) => v.matchas((w: any) => w.name("a")); // a capture pattern, Python Patterns' own: its Term lists no Python3 kind
  if (dialect === SVPP.DIALECT) return (v: any) => v.macrousage((w: any) => w.name((i: any) => i.spelling("a"))); // a macro's usage, the Preprocessor's one kind yet
  if (FAMILY.includes(dialect)) { // SystemVerilog's nameexpression, holding its identifier, in each of its family's unions
    return (v: any) => v.nameexpression((w: any) => w.name((n: any) => n.identifier((i: any) => i.spelling("a"))));
  }
  return (v: any) => v.variable((w: any) => w.name("a"));
}
const error = (dialect: F.Dialect) => (u: any) => u.error((r: any) => r.text("(a +* 1)").terms([ // what a parser skipped
  dialect.spec(recovered(dialect))])); // and the terms it recovered, of its dialect
const missing = (u: any) => u.missing((r: any) => r.kind("expression")); // what it expected
for (const dialect of DIALECTS) {
  const [skipped, expected] = [term(error(dialect), dialect), term(missing, dialect)];
  assert(F.form(skipped).kind === "error" && equal(F.form(skipped).attributes, map({ text: "(a +* 1)" })));
  assert(F.form(skipped).arguments.length === 1 && form_equal(F.form(expected), new F.Form("missing", { kind: "expression" })));
  assert(same(dialect.validate(skipped), ["invalid syntax '(a +* 1)'"]) && same(dialect.validate(expected), ["missing expression"]));
  assert(same(dialect.validate(term((u: any) => u.error((r: any) => r), dialect)), ["invalid syntax"]));
  assert(same(dialect.validate(term((u: any) => u.missing((r: any) => r), dialect)), ["missing a term"]));
  assert(F.same(term(dialect.make(F.form(skipped)), dialect), skipped)); // forms and make, as for every kind
  assert(same([skipped, expected].map((t) => dialect.kind(t)[2].role), [F.INVALID, F.INVALID]));
  const simplify = SIMPLIFIERS.get(dialect);
  if (simplify !== undefined) { // invalid syntax stays, named: it has no value of its own
    for (const [invalid, kind] of [[skipped, "error"], [expected, "missing"]] as const) {
      const first = simplify.run(invalid).residues[0] as FV.Residue; // then what it recovered, where no rule folds it
      assert(first.reason === `${dialect.name()} ${kind} is not simplified` && first.category === "b");
    }
  }
  const print = PRINTERS.get(dialect);
  if (print !== undefined) { // an error as the source it spans, a missing term as nothing
    assert(print(skipped) === (dialect === X.DIALECT ? "=(a +* 1)" : "(a +* 1)"));
    assert(print(expected) === (dialect === X.DIALECT ? "=" : ""));
    raises(ValueError, () => print(term((u: any) => u.error((r: any) => r), dialect)), `${dialect.name()} error has no text to print`);
  }
}
assert(Python3.Text.ToText(term(P.binop("+", P.name("a"), missing), P.DIALECT)) === "a + "); // within a term, as written
assert(same(sorted(SY.free(term(error(E.DIALECT)), E.DIALECT)), ["a"])); // the terms recovered within an error are terms
raises(ValueError, () => Translators.between(P.DIALECT, M.DIALECT).forward(term(error(P.DIALECT), P.DIALECT)),
  "Python3 error has no MATLAB counterpart"); // nothing translates invalid syntax
raises(ValueError, () => E.DIALECT.infer(term(missing)), "cannot infer the domain of an invalid expression: missing expression");
// Every dialect's Type and Pattern hold invalid syntax too: validation reports it within a type or a target, printers
// write it as written, evaluation refuses it by name
for (const union of ["Python.Python3.Pattern", "TypeScript.Types.Term", "TypeScript.Patterns.Term", "C.Types.Term",
  "C.Expressions.Pattern", "Cpp.Types.Term", "Cpp.Expressions.Pattern"]) {
  const names = store.registered(`Expressions.${union}`).branches.map((b: any) => b.name);
  assert([...INVALID].every((tag) => names.includes(tag)), union);
}
{
  const one = (v: any) => v.literal((li: any) => li.value((n: any) => n.as_scalar(1n))); // Python3's and C's
  const named = term((u: any) => u.namedexpr((r: any) => r.target((t: any) => t.error((e: any) => e.text("1"))).value(one)), P.DIALECT); // (1 := 1)
  assert(same(P.DIALECT.validate(named), ["target: invalid syntax '1'"]) && Python3.Text.ToText(named) === "(1 := 1)");
  const generator = term(P.generatorexp(P.name("p"), P.comprehension((t: any) => t.missing((m: any) => m.kind("identifier")), P.list_(1n))), P.DIALECT);
  assert(same(P.DIALECT.validate(generator, ["p"]), ["generators 0 target: missing identifier"]) && Python3.Text.ToText(generator) === "(p for  in [1])");
  raises(NotEvaluated, () => Python3.Simplifiers.OfAny(generator).evaluate(), "Python3 missing is not simplified");
  const cast = term((u: any) => u.tsasexpression((r: any) => r.expression(tsPerson).typeAnnotation((t: any) => t.missing((m: any) => m.kind("type")))), TS.DIALECT);
  assert(same(TS.DIALECT.validate(cast, ["person"]), ["typeAnnotation: missing type"]) && TypeScript5.Text.ToText(cast) === "person as ");
  const arrow = term((u: any) => u.arrowfunctionexpression((r: any) => r.params([(p: any) => p.error((e: any) => e.text("a b"))])), TS.DIALECT);
  assert(same(TS.DIALECT.validate(arrow), ["argument 0: invalid syntax 'a b'"]) && TypeScript5.Text.ToText(arrow) === "(a b) => {}");
  for (const [dialect, printer] of [[C.DIALECT, CDialect.Text.ToText], [CPP.DIALECT, CppDialect.Text.ToText]] as [F.Dialect, (e: unknown) => string][]) {
    const typed = term((u: any) => u.castexpression((r: any) => r.type((t: any) => t.error((e: any) => e.text("int int"))).operand(one)), dialect);
    assert(same(dialect.validate(typed), ["type: invalid syntax 'int int'"]) && printer(typed) === "(int int)1"); // its type a Types term
  }
}

<a id="FRM-12"></a>
## FRM-12 · A dialect reads only its own terms, and those of the dialects it reads: a term holds its dialect, a shared kind's subterm may be any dialect's term, and validation reports another's; C and C++ each define the kinds they share; SystemVerilog's six dialects read each other's

In [ ]:
const foreign = term(P.let_("a", E.variable("a"), P.name("a")), P.DIALECT); // a shared kind's subterm: Patterns' variable in it
assert(F.dialectOf(F.form(foreign).arguments[0]) === E.DIALECT.union && F.dialectOf(foreign) === P.DIALECT.union);
assert(same(P.DIALECT.validate(foreign), ["value: a Patterns variable is not a Python3 term"]));
raises(ValueError, () => Python3.Simplifiers.OfAny(foreign).evaluate(), "a Patterns variable is not a Python3 term");
raises(ValueError, () => Matlab.Text.ToText(term(X.function("AND", true), X.DIALECT)), "an Excel call is not a MATLAB term");
assert(!F.same(term(E.variable("a")), term(P.name("a"), P.DIALECT))); // the same kind, in another dialect
const own = term(P.binop("+", P.name("a"), 1n), P.DIALECT); // a dialect's own kind's subterm: its union's terms alone
assert(same(F.form(own).arguments.map((a) => F.dialectOf(a)), [P.DIALECT.union, P.DIALECT.union]));
assert((F.form(own).arguments[0] as any).as_schema().name === P.DIALECT.union); // one level: its kind alone
raises(Error, () => term(P.binop("+", X.cell("A1"), 1n), P.DIALECT), "'cell' is not a member of a union value"); // the schema refuses a kind its union lacks
// C and C++: each its own kinds, of the same shapes, and its own Term (the standards that have each are data: FRM-13)
raises(TypeError, () => CPP.DIALECT.spec(new Uint8Array([120])), "expected a term, a writer or a native value, got b'x'"); // no literal of C++ holds bytes
assert(CDialect.Text.ToText(samples.C) === ( // every kind, a body of statements written `{}`
  '{!person.a && person->b[0] > 1.5f ? (uint8_t)person(1) : 5u, nullptr, "a" "b", (person), _Generic(person, int: 1, d'
  + 'efault: 0), person++, sizeof(int), _Alignof(person), person += 1, .x[0] = person, (int[]){1}, ({}), __extension__ person}'));
assert(CppDialect.Text.ToText(samples.Cpp) === (
  '{!person.a && person->b[0] > 1.5f ? (uint8_t)person(1) : 5u, nullptr, 10_km, "a" R"(b)", std::cout, this, (person), '
  + '[=, &person](auto y) {}, (person + ...), requires (int z) { z + 1; }, int(person), person--, static_cast<int>(person'
  + '), typeid(person), co_await person, sizeof(int), sizeof...(Ts), alignof(int), noexcept(person), new int{1}, delete['
  + '] person, person = 0, throw, co_yield person, person..., .x = person, ({}), __extension__ person}'));
assert(CDialect.Simplifiers.OfAny(term(C.binary("+", 1n, 2n), C.DIALECT)).evaluate() === 3n
  && CppDialect.Simplifiers.OfAny(term(CPP.binary("+", 1n, 2n), CPP.DIALECT)).evaluate() === 3n);
raises(Error, () => term(C.binary("+", E.variable("x").add(1n), 1n), C.DIALECT), "'call' is not a member of a union value"); // C's own kinds hold C's terms alone
// SystemVerilog's dialects read each other's terms (`reads`): a cast's type is a Types term, read by SystemVerilog as
// Types reads it, and a dimension's bound an Expressions term, read by Types as SystemVerilog reads it; any other
// dialect's term is refused as before, a sublanguage's named by its family and its own name.
const svCast = term(SV.cast(svAtom("int"), svPerson), SV.DIALECT);
const svTyped = F.form(svCast).arguments[0];
assert(F.dialectOf(svTyped) === SVT.DIALECT.union && SV.DIALECT.kind(svTyped)[2] === SVT.DIALECT.kinds().get("integeratomtype"));
assert(same(SV.DIALECT.validate(svCast, ["person"]), []) && same(SVT.DIALECT.validate(samples["SystemVerilog Types"], ["person"]), []));
assert(same(SV.DIALECT.validate(term(SV.cast((t: any) => t.integeratomtype((a: any) => a.keyword("word")), 1n), SV.DIALECT)), [
  "type: IntegerAtomType.keyword cannot be 'word'"])); // its checks, as Types reads it
assert(same(C.DIALECT.validate(svTyped), ["a SystemVerilog Types integeratomtype is not a C term"]));
assert(same(SV.DIALECT.validate(term(E.variable("a"))), ["a Patterns variable is not a SystemVerilog term"]));
raises(ValueError, () => E.DIALECT.kind(svTyped), "a SystemVerilog Types integeratomtype is not a Patterns term");

<a id="FRM-13"></a>
## FRM-13 · Standards are data: each dialect covers its editions by designation, each found by its designation or its common names, and records the kinds it has

In [ ]:
// Each dialect's define writes the editions it covers, owned by their designations as their standards bodies write
// them, each holding its formal reference's parts and its common names: the tags it is found by, either kind of reference.
for (const define of [Python3.define, TypeScript5.define]) store.transaction(define); // their versions, beside their domains
const [c, cpp] = [store.Instances.Expressions.C.Expressions, store.Instances.Expressions.Cpp.Expressions];
const [py3, ts5] = [store.Instances.Expressions.Python.Python3, store.Instances.Expressions.TypeScript.TypeScript5];
assert(same([...FS.standards(c).keys()], ["ISO/IEC 9899:2011", "ISO/IEC 9899:2018", "ISO/IEC 9899:2024"]));
const c11 = c.standards({ designation: "ISO/IEC 9899:2011" }); // by its path: its dialect's, then its designation
assert(FS.standards(c).get("ISO/IEC 9899:2011") === c11 && FS.designationOf(c11) === "ISO/IEC 9899:2011");
assert(same(FS.tags(c11), ["ISO/IEC 9899:2011", "C11"]) && FS.find(c, "C11") === c11 && FS.find(c, "ISO/IEC 9899:2011") === c11);
assert(FS.find(c, "C++17") === null && FS.find(cpp, "C11") === null); // each dialect's own editions
assert(equal(FS.properties(c11), map({ bodies: ["org.iso", "ch.iec"], number: "9899", revision: ["year", 2011n],
  names: ["C11"] }))); // a joint standard, by its bodies' reverse-domain names
assert(equal([...FS.standards(c).values()].map((s) => FS.tags(s)), [
  ["ISO/IEC 9899:2011", "C11"], ["ISO/IEC 9899:2018", "C17"], ["ISO/IEC 9899:2024", "C23"]]));
assert(equal([...FS.standards(cpp).values()].map((s) => FS.tags(s)), [
  ["ISO/IEC 14882:2017", "C++17"], ["ISO/IEC 14882:2020", "C++20"], ["ISO/IEC 14882:2024", "C++23"]]));
assert(equal(FS.properties(FS.find(cpp, "C++23")!).get("revision"), ["year", 2024n])); // ISO/IEC 14882:2024 is C++23
const VERSIONS = Array.from({ length: 16 }, (_, minor) => `Python 3.${minor}`); // each by its published name, no common one
assert(same([...FS.standards(py3).keys()], VERSIONS) && same(FS.tags(FS.find(py3, "Python 3.12")!), ["Python 3.12"]));
assert(equal(FS.properties(FS.find(py3, "Python 3.12")!), map({ bodies: ["org.python"], revision: ["version", "3.12"] })));
const TS_VERSIONS = Array.from({ length: 10 }, (_, minor) => `TypeScript 5.${minor}`);
assert(same([...FS.standards(ts5).keys()], TS_VERSIONS));
assert(equal(FS.properties(FS.find(ts5, "TypeScript 5.0")!), map({ bodies: ["com.microsoft"], revision: ["version", "5.0"] })));
// Each kind records the editions that have it, as data: each edition its kinds, by their schemas, and a kind the
// editions of a dialect it is in; an extension and invalid syntax are in none.

/** Each of a dialect's kinds, by its tag, and the editions that have it, by their last tags: their common names, or
 * their designations where they have none. */
function editions(dialect: F.Dialect, singleton: any): Map<string, string[]> {
  const union = store.registered(dialect.union);
  return new Map(union.branches.map((b: any) => [b.name, [...FS.of(singleton, b.schema).values()].map(
    (s) => FS.tags(s)[FS.tags(s).length - 1] as string)]));
}

assert(same([...C.EXTENSIONS], ["statementexpression", "extensionexpression"]) && CPP.EXTENSIONS === C.EXTENSIONS); // GNU's
const READ_KINDS: [string, string[]][] = [["typeid", []], ["definedexpression", []]]; // C Types' and C Preprocessor's kinds, in
// their own dialects' editions
assert(equal(editions(C.DIALECT, c), new Map([...[...C.DIALECT.kinds().keys()].map((tag) => [tag, C.EXTENSIONS.has(tag)
  || INVALID.has(tag) ? [] : tag === "constant" ? ["C23"] : ["C11", "C17", "C23"]] as [string, string[]]), ...READ_KINDS]))); // nullptr:
// C23's; C17 is C11's corrigenda
const since20 = ["requiresexpression", "awaitexpression", "yieldexpression", "designatedinitializer"];
assert(equal(editions(CPP.DIALECT, cpp), new Map([...[...CPP.DIALECT.kinds().keys()].map((tag) => [tag,
  CPP.EXTENSIONS.has(tag) || INVALID.has(tag) ? [] : since20.includes(tag) ? ["C++20", "C++23"] : ["C++17", "C++20", "C++23"]] as
  [string, string[]]), ...READ_KINDS])));
assert(same([...TS.EXTENSIONS], ["jsxelement", "jsxfragment", "jsxemptyexpression"])); // JSX: every other kind in every version
const tsUnion = (dialect: F.Dialect): string[] => store.registered(dialect.union).branches.map((b: any) => b.name);
assert(equal(editions(TS.DIALECT, ts5), new Map(tsUnion(TS.DIALECT).map((tag) => [tag, // the patterns its `Term` lists,
  TS.EXTENSIONS.has(tag) || INVALID.has(tag) || TSP.TAGS.includes(tag) ? [] : TS_VERSIONS])))); // Patterns' own
const [tst, tsp] = [store.Instances.Expressions.TypeScript.Types, store.Instances.Expressions.TypeScript.Patterns];
for (const [dialect, singleton] of [[TST, tst], [TSP, tsp]] as const) { // TypeScript's types and patterns cover its
  // versions, each its own kinds
  assert(same([...FS.standards(singleton).keys()], TS_VERSIONS) && dialect.SINCE.size === 0);
  assert(equal(FS.properties(FS.find(singleton, "TypeScript 5.9")!), map({ bodies: ["com.microsoft"], revision: ["version", "5.9"] })));
  assert(equal(editions(dialect.DIALECT, singleton), new Map(tsUnion(dialect.DIALECT).map((tag) => [tag,
    dialect.TAGS.includes(tag) ? TS_VERSIONS : []]))));
}
const python = editions(P.DIALECT, py3); // each kind from the version it arrived in
assert(same(python.get(P.BINOP)!, VERSIONS) && same(python.get(P.NAMEDEXPR)!, VERSIONS.slice(8))
  && same(python.get(P.YIELDFROM)!, VERSIONS.slice(3)));
assert(same(python.get(P.TEMPLATESTR)!, ["Python 3.14", "Python 3.15"]) && python.get(P.AWAIT)![0] === "Python 3.5");
assert(python.get(P.JOINEDSTR)![0] === "Python 3.6");
assert(same([...P.EXTENSIONS], ["let", "import", "importfrom"])
  && [...P.EXTENSIONS, ...INVALID].every((tag) => python.get(tag)!.length === 0));
const [pt_, pp_] = [store.Instances.Expressions.Python.Types, store.Instances.Expressions.Python.Patterns]; // Python's sublanguages
assert(same([...FS.standards(pt_).keys()], VERSIONS) && same([...FS.standards(pp_).keys()], VERSIONS.slice(10))); // its own kinds alone
assert(equal(editions(PYT.DIALECT, pt_), new Map(store.registered(PYT.DIALECT.union).branches.map((b: any) => [b.name,
  PYT.TAGS.includes(b.name) ? VERSIONS.slice(12) : []]))));
assert(equal(editions(PYP.DIALECT, pp_), new Map([...PYP.DIALECT.kinds().keys()].map((tag) => [tag, PYP.TAGS.includes(tag) ? VERSIONS.slice(10) : []]))));
// The regular expression dialects: Python's covers Python 3.0 to 3.15, `atomicgroup` from 3.11; TypeScript's ECMA-262's
// editions, by Ecma's designations, the v flag's kinds from ES2024, modifiers from ES2025, ES2026 adding none; C++'s
// C++'s editions, every kind in each.
const pr_ = store.Instances.Expressions.Python.RegExp;
assert(same([...FS.standards(pr_).keys()], VERSIONS) && equal(editions(PYR.DIALECT, pr_), new Map([...PYR.DIALECT.kinds().keys()].map(
  (tag) => [tag, tag === "atomicgroup" ? VERSIONS.slice(11) : PYR.TAGS.includes(tag) ? VERSIONS : []]))));
const tsr_ = store.Instances.Expressions.TypeScript.RegExp;
const LAST = TSR.EDITIONS.map(([, , names]) => names.at(-1)!); // ES5.1, ES2015 ... ES2026
assert(same([...FS.standards(tsr_).keys()], TSR.EDITIONS.map((edition) => edition[0])) && LAST.at(-1) === "ES2026");
assert(same(FS.tags(FS.find(tsr_, "ES2015")!), ["ECMA-262 6th Edition", "ECMAScript 2015", "ES2015"]));
assert(same(FS.tags(FS.find(tsr_, "ES2026")!), ["ECMA-262 17th Edition", "ECMAScript 2026", "ES2026"]));
assert(equal(editions(TSR.DIALECT, tsr_), new Map(tsUnion(TSR.DIALECT).map((tag) => [tag,
  TSR.TAGS.includes(tag) ? LAST.slice(TSR.SINCE.get(tag) ?? 0) : []]))));
assert(same(FS.kinds(FS.find(tsr_, "ES2026")!), FS.kinds(FS.find(tsr_, "ES2025")!))); // its pattern grammar unchanged
const xr_ = store.Instances.Expressions.Cpp.RegExp;
assert(equal([...FS.standards(xr_).values()].map((s) => FS.tags(s)), [...FS.standards(cpp).values()].map((s) => FS.tags(s))));
assert(equal(editions(XR, xr_), new Map([...XR.kinds().keys()].map((tag) => [tag,
  CppRegExp.Expressions.TAGS.includes(tag) ? ["C++17", "C++20", "C++23"] : []]))));
// POSIX's two cover POSIX.1's editions, by IEEE's designations, the ISO/IEC designations C++ cites among their common
// names, every kind in each.
[PosixBRE, PosixERE].forEach((pkg: any, i) => {
  const dialect = POSIX[i] as F.Dialect;
  const singleton = store.singleton(`Expressions.${dialect.path}`);
  assert(same([...FS.standards(singleton).keys()], [2001, 2008, 2017, 2024].map((year) => `IEEE Std 1003.1-${year}`)));
  assert(equal(editions(dialect, singleton), new Map([...dialect.kinds().keys()].map((tag) => [tag, pkg.Expressions.TAGS.includes(tag)
    ? ["ISO/IEC 9945:2003", "ISO/IEC/IEEE 9945:2009", "POSIX.1-2017", "POSIX.1-2024"] : []]))));
});
assert(same(FS.kinds(FS.find(c, "C23")!), C_UNION.branches.filter((b: any) => !C.EXTENSIONS.has(b.name) && !INVALID.has(b.name)
  && !READ_KINDS.some(([tag]) => tag === b.name))
  .map((b: any) => b.schema)));
// A kind's schema reads the same entries from its side, through its extension adjacency `standards`.
const namedexpr = store.registered(P.DIALECT.union).branches.find((b: any) => b.name === P.NAMEDEXPR).schema;
assert(same([...new Builders.OfObject.Builder(namedexpr, false, store).standards].map((e: any) => FS.designationOf(e.standard)),
  VERSIONS.slice(8)));
// SystemVerilog's six dialects cover the IEEE's editions, each with its own kinds: Expressions, Types and the Preprocessor
// Verilog's (IEEE 1364) and SystemVerilog's (IEEE 1800, which merged it), Patterns, Assertions and Constraints IEEE
// 1800's alone; each kind from the edition it arrived in, as mbse-programs' SINCE has it.
const VERILOG = [1995, 2001, 2005].map((year) => `IEEE Std 1364-${year}`);
const IEEE1800 = [2005, 2009, 2012, 2017, 2023].map((year) => `IEEE Std 1800-${year}`);
const LABELS = [...[1995, 2001, 2005].map((year) => `Verilog-${year}`), ...[2005, 2009, 2012, 2017, 2023].map((year) => `SystemVerilog-${year}`)];
assert(same(SV.EDITIONS.map((e) => e[0]), [...VERILOG, ...IEEE1800]) && same(SV.EDITIONS.map((e) => e[3]), LABELS));
const singletons = new Map(FAMILY.map((dialect) => [dialect, store.singleton(`Expressions.${dialect.path}`)]));
for (const [dialect, singleton] of singletons) {
  const whole = [SV.DIALECT, SVT.DIALECT, SVPP.DIALECT].includes(dialect);
  assert(same([...FS.standards(singleton).keys()], whole ? [...VERILOG, ...IEEE1800] : IEEE1800), dialect.name());
}
const svSingleton = singletons.get(SV.DIALECT);
assert(same(FS.tags(FS.find(svSingleton, "SystemVerilog-2023")!), ["IEEE Std 1800-2023", "SystemVerilog-2023"]));
assert(equal(FS.properties(FS.find(svSingleton, "Verilog-2005")!), map({ bodies: ["org.ieee"], number: "1364", revision: ["year", 2005n],
  names: ["Verilog-2005"] }))); // Verilog, IEEE 1364, its subset
for (const [dialect, own] of [[SV.DIALECT, SV.TAGS], [SVT.DIALECT, SVT.TAGS], [SVP.DIALECT, SVP.TAGS], [SVA.DIALECT, SVA.TAGS],
  [SVC.DIALECT, SVC.TAGS], [SVPP.DIALECT, SVPP.TAGS]] as [F.Dialect, readonly string[]][]) { // each edition its own kinds alone
  const whole = [SV.DIALECT, SVT.DIALECT, SVPP.DIALECT].includes(dialect);
  const since = new Map([...SV.SINCE, ...SVT.SINCE, ...SVA.SINCE, ...SVC.SINCE]); // a kind not listed is in every edition its dialect covers; the
  assert(equal(editions(dialect, singletons.get(dialect)), new Map(store.registered(dialect.union).branches.map((b: any) => [b.name,
    own.includes(b.name) ? LABELS.slice(since.get(b.name) ?? (whole ? 0 : 3)) : []]))), dialect.name()); // dialects it reads, in theirs
}
assert(same(editions(SV.DIALECT, svSingleton).get("castexpression")!, LABELS.slice(3))
  && same(editions(SVT.DIALECT, singletons.get(SVT.DIALECT)).get("structtype")!, LABELS.slice(3)));
assert(same(editions(SVPP.DIALECT, singletons.get(SVPP.DIALECT)).get("macrousage")!, LABELS));
assert(store.transaction(SVTypes.define)[0].get("SystemVerilog_2023") === FS.find(singletons.get(SVT.DIALECT), "SystemVerilog-2023"));
// Every other dialect covers none: its code names no standard.
assert(["Patterns.Expressions", "MATLAB.Expressions", "Excel.Expressions", "LaTeX.Expressions"]
  .every((path) => FS.standards(store.singleton(`Expressions.${path}`)).size === 0));
// Run again, a define writes nothing; an edition validates. The model: an edition a reference object its dialect owns
// by designation, its revision a year, an edition or a version, and a set relating editions to kinds' schemas.
assert(store.transaction(CDialect.define)[0].get("C11") === c11
  && store.transaction(CppDialect.define)[0].get("Cpp17") === FS.find(cpp, "C++17"));
const Standard = store.registered("Expressions.Framework.Standard");
assert(Validators.Validate(store)(Standard, c11).length === 0);
assert(Standard.ref && same([...Standard.properties.keys()], ["bodies", "number", "revision", "names"]));
assert(same([...Standard.adjacencies.keys()], ["dialect", "kinds"]) && Standard.adjacencies.get("dialect").owner);
assert(same(store.registered("Expressions.Framework.Revision").branches.map((b: any) => b.name), ["year", "edition", "version"]));
const [Covers, HasStandard] = [store.registered("Expressions.Framework.Covers"), store.registered("Expressions.Framework.HasStandard")];
assert(equal(Covers.uniques.map((u: any) => [...u]), [["dialect", "designation"], ["standard"]]));
assert(same([...HasStandard.entry.links.keys()], ["kind", "standard"]) && equal(HasStandard.uniques.map((u: any) => [...u]), [[]])); // a set
assert(store.registered("Schemas.OfObject").adjacencies.has("standards"));
// Until a dialect covers it, an edition has no designation, and is found by its common names alone.
const loose = store.transaction(() => {
  const made = store.Expressions.Framework.Standard((s: any) => s.names(["loose"]));
  assert(FS.designationOf(made) === null && same(FS.tags(made), ["loose"]));
  store.Expressions.Patterns.Expressions.Dialect(store.singleton("Expressions.Patterns.Expressions"),
    (b: any) => b.standards({ "Loose 1": (e: any) => e.standard(made) }));
  return made;
});
assert(same(FS.tags(loose), ["Loose 1", "loose"]) && FS.find(store.singleton("Expressions.Patterns.Expressions"), "loose") === loose);